# LAB: Practice with Tavily + LangChain Agents

### Objective
Reinforce your understanding of LangChain agents integrated with Tavily for real-time web search by solving two open-ended tasks. You'll:
- Load and configure a LangChain agent with Tavily
- Build effective prompts
- Generate structured, useful outputs from real-time data

In [3]:
%pip install langchain-classic

   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/1.0 MB 8.4 MB/s  0:00:00

   -------------------- ------------------- 1/2 [langchain-classic]
   -------------------- ------------------- 1/2 [langchain-classic]
   -------------------- ------------------- 1/2 [langchain-classic]
   -------------------- ------------------- 1/2 [langchain-classic]
   -------------------- ------------------- 1/2 [langchain-classic]
   -------------------- ------------------- 1/2 [langchain-classic]
   -------------------- ------------------- 1/2 [langchain-classic]
   -------------------- ------------------- 1/2 [langchain-classic]
   -------------------- ------------------- 1/2 [langchain-classic]
   -------------------- ------------------- 1/2 [langchain-classic]
   -------------------- ------------------- 1/2 [langchain-classic]
   -------------------- ------------------- 1/2 [langchain-classic]
   -------------------- -----------------

In [5]:
%pip install -U langchain-openai

Note: you may need to restart the kernel to use updated packages.


In [3]:
%pip install -U langchain-community tavily-python

   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ----------------------------------- ---- 2.1/2.4 MB 13.2 MB/s eta 0:00:01
   ---------------------------------------- 2.4/2.4 MB 10.1 MB/s  0:00:00

   -------------------------- ------------- 2/3 [langchain-community]
   -------------------------- ------------- 2/3 [langchain-community]
   -------------------------- ------------- 2/3 [langchain-community]
   -------------------------- ------------- 2/3 [langchain-community]
   -------------------------- ------------- 2/3 [langchain-community]
   -------------------------- ------------- 2/3 [langchain-community]
   -------------------------- ------------- 2/3 [langchain-community]
   -------------------------- ------------- 2/3 [langchain-community]
   -------------------------- ------------- 2/3 [langchain-community]
   -------------------------- ------------- 2/3 [langchain-community]
   -------------------------- ------------- 2/3 [langchain-community]
   ------

### Setup

In [20]:
import os
from langchain_classic.agents import initialize_agent, AgentType, AgentExecutor
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
import tiktoken
from IPython.display import Markdown, display
from  dotenv import load_dotenv
import warnings
warnings.filterwarnings('ignore')
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.tools import Tool
from langchain.agents import create_agent

load_dotenv()
tavily_search = TavilySearchResults()

# LLM + Encoding
llm = ChatOpenAI(model="gpt-6-luna")
encoding = tiktoken.encoding_for_model("gpt-4")

# Safe Tavily wrapper
def safe_search(query: str) -> str:
    result = tavily_search.run(query)

    # Ensure result is a string — Tavily returns dict with 'snippets' sometimes
    if isinstance(result, dict):
        result_text = result.get("content", "") or str(result)
    else:
        result_text = str(result)

    tokens = encoding.encode(result_text)
    trimmed = encoding.decode(tokens[:3500])  # leave room for GPT-4 response
    return trimmed


# LangChain tool
tools = [Tool(name="TavilySafeSearch", func=safe_search, description="Web search tool")]
agent = create_agent(model=llm, tools=tools, system_prompt="You are a research assistant. Use web search to find reliable information and provide structured Markdown reports with source URLs.")

### Exercise 1: AI in Healthcare

Goal: Investigate and summarize the latest advancements in generative AI applied to healthcare in 2025.

- Design a prompt that asks the agent to retrieve the most recent updates.
- Ensure the agent outputs a structured response in Markdown.

In [ ]:
# Your prompt here
prompt_1 = """
Research the most significant and recent developments in
generative AI applied to healthcare during 2025.

Use the web search tool to retrieve reliable, up-to-date information.
Focus exclusively on developments published between
January 1, 2025, and December 31, 2025.

Cover the following topics:
1. Clinical applications of generative AI
2. AI-assisted diagnosis and medical imaging
3. Drug discovery and pharmaceutical research
4. Generative AI in patient care and hospital administration
5. Regulatory developments, ethical concerns, and patient privacy

Return your findings as a structured Markdown report
with the following sections:

# Generative AI in Healthcare: 2025 Developments

## 1. Executive Summary
Summarize the 3–5 most important developments.

## 2. Key Developments
For each development, include:
- Title
- Date
- Description
- Organizations involved
- Potential impact on healthcare
- Source URL

## 3. Emerging Trends
Identify the main trends observed during 2025.

## 4. Challenges and Risks
Discuss limitations, patient safety, bias, privacy,
and regulatory challenges.

## 5. Conclusion
Summarize the overall progress and future implications.

## 6. References
Provide a Markdown list of sources with URLs.

Important instructions:
- Prioritize credible sources, including peer-reviewed
  studies, medical institutions, regulatory agencies,
  and established technology publications.
- Distinguish between clinical validation, pilot projects,
  research announcements, and deployed applications.
- Include specific dates wherever available.
- Do not invent information or references.
- If evidence is insufficient, explicitly state this.
- Format the entire response for display."""

# Run it
response_1 = agent.invoke({"messages": [
        {"role": "user", "content": prompt_1}
    ]
})


In [30]:
from IPython.display import Markdown, display

last_message = response_1["messages"][-1]

final_answer = last_message.content

if isinstance(final_answer, list):
    final_answer = "\n".join(
        block["text"]
        for block in final_answer
        if isinstance(block, dict) and block.get("type") == "text"
    )

display(Markdown(final_answer))

# Generative AI in Healthcare: 2025 Developments

*Scope: developments published or formally announced from January 1 through December 31, 2025. The report distinguishes clinical trials and reader studies from pilots, research models, and routine deployment. Results from a study published in 2025 may involve data collected earlier.*

## 1. Executive Summary

- **Generative AI gained early clinical-trial evidence in mental health.** Dartmouth’s randomized study of the research chatbot Therabot reported symptom improvements, but it was a limited study—not evidence that consumer chatbots can safely replace clinicians.
- **Medical-imaging AI moved from benchmark demonstrations toward reader studies.** 2025 studies of AI-generated chest X-ray reports found potential workflow benefits, while also underscoring the need for radiologist review and broader validation.
- **AI-assisted drug discovery reached an important clinical milestone.** Rentosertib, a drug whose target and molecule were discovered and designed using AI, reported Phase IIa results for idiopathic pulmonary fibrosis. The result is promising but preliminary, and requires larger, longer trials.
- **Health systems and researchers tested generative AI in everyday workflows.** Ambient scribes received randomized-trial evaluation, while patient-message response systems and open medical models offered further research and development pathways.
- **Oversight focused increasingly on lifecycle evidence and governance.** FDA actions addressed AI evidence in drug decisions, medical-device performance, and mental-health devices; the EU AI Act’s phased obligations and Joint Commission–CHAI guidance added governance expectations.

## 2. Key Developments

### 1. Therabot: a randomized trial of a generative AI mental-health chatbot

- **Date:** March 27, 2025
- **Description:** Dartmouth researchers published a randomized trial of Therabot, an expert-developed generative AI chatbot informed by evidence-based psychotherapies. The trial randomized 210 adults; 106 received Therabot and the remainder were assigned to a wait-list control. The researchers reported greater symptom reductions in the Therabot group across depression, anxiety, and eating-disorder concerns. Dartmouth described reductions within the Therabot group of 51% for depression, 31% for anxiety, and 19% for eating-disorder concerns.
- **Organizations involved:** Dartmouth College and its Geisel School of Medicine’s Center for Technology and Behavioral Health; NEJM AI.
- **Potential impact on healthcare:** A purpose-built chatbot could offer structured support between appointments or where access to mental-health care is limited. The study provides rare randomized evidence for a generative AI mental-health intervention.
- **Evidence status and caveats:** This was a controlled research trial, not broad clinical deployment. It used a wait-list comparator, and commentators questioned whether a stronger attention-matched comparator would better establish the treatment effect. Dartmouth researchers cautioned that no generative AI agent is ready to operate autonomously in mental-health care.
- **Source URL:** [NEJM AI trial](https://ai.nejm.org/doi/abs/10.1056/AIoa2400802); [Dartmouth study summary](https://home.dartmouth.edu/news/2025/03/first-therapy-chatbot-trial-yields-mental-health-benefits); [NEJM AI commentary](https://ai.nejm.org/doi/abs/10.1056/AIp2500453)

### 2. Generative AI-assisted chest X-ray reporting evaluated in reader studies

- **Date:** March 11 and March 25, 2025
- **Description:** Two studies published in *Radiology* assessed systems that generate preliminary chest X-ray reports. In one reader study involving five radiologists and 758 X-rays, radiologists using an AI-generated report took less time per image and showed improved performance on some reporting measures. A separate study evaluated a domain-specific multimodal model, trained on a large radiograph-report dataset, and reported that many generated reports were accepted by radiologists without modification.
- **Organizations involved:** Researchers at Mass General Brigham and other collaborating institutions; Soombit.ai was associated with the AIRead model in the March 11 report.
- **Potential impact on healthcare:** Draft reports could reduce the time needed for routine image interpretation and help radiologists manage high workloads. This may be particularly useful as a drafting or second-reader tool.
- **Evidence status and caveats:** These were reader studies, not proof of improved patient outcomes or routine autonomous interpretation. Report acceptance and speed do not establish safety across hospitals, patient populations, or uncommon findings; generated text still requires clinician review.
- **Source URLs:** [March 11 *Radiology* study](https://pubs.rsna.org/doi/10.1148/radiol.241646); [March 25 *Radiology* study](https://pubs.rsna.org/doi/10.1148/radiol.250234)

### 3. Rentosertib reported Phase IIa results after AI-supported discovery and design

- **Date:** June 3, 2025
- **Description:** A randomized, double-blind, placebo-controlled Phase IIa trial of rentosertib (formerly ISM001-055) for idiopathic pulmonary fibrosis was published in *Nature Medicine*. The drug’s target, TNIK, was identified using AI, and the molecule was designed and optimized with generative AI. Among 71 trial participants, the 60 mg once-daily group had a mean forced vital capacity change of **+98.4 mL** at 12 weeks, compared with **−20.3 mL** in the placebo group. The authors described the findings as warranting further investigation.
- **Organizations involved:** Insilico Medicine and clinical collaborators, including investigators in China.
- **Potential impact on healthcare:** The result is a notable clinical-stage milestone for AI-enabled drug discovery: it links AI-assisted target identification and molecular design to a candidate tested in a controlled human trial.
- **Evidence status and caveats:** Phase IIa is early clinical evidence, not proof of durable benefit or regulatory approval. The trial was small and short; the publication calls for larger, longer studies. The sponsor’s involvement should also be considered when interpreting company communications about the findings.
- **Source URL:** [Peer-reviewed *Nature Medicine* study](https://www.nature.com/articles/s41591-025-03743-2); [Insilico announcement](https://www.insilico.com/news/tnrecuxsc1-insilico-announces-nature-medicine-publi)

### 4. Ambient AI scribes received randomized clinical evaluation

- **Date:** November 26, 2025
- **Description:** A pragmatic randomized trial compared Microsoft Dragon Ambient eXperience (DAX) Copilot and Nabla with usual care among 238 outpatient physicians across 14 specialties. The study analyzed about 72,000 patient encounters. Nabla users had a statistically significant **9.5% greater reduction in note-writing time** than the control group; the corresponding change for DAX did not reach statistical significance on the primary measure. Clinicians also reported occasional clinically significant inaccuracies.
- **Organizations involved:** UCLA Health researchers; Microsoft; Nabla; NEJM AI.
- **Potential impact on healthcare:** Ambient scribes could reduce time spent documenting visits and give clinicians more attention for patients. Product-specific differences in the trial emphasize the importance of evaluating tools individually rather than assuming equivalent benefits.
- **Evidence status and caveats:** This is randomized, real-world workflow evidence, but it does not establish that all scribes improve outcomes or reliably reduce burnout. Clinicians must review and correct drafts before they become part of the medical record.
- **Source URL:** [Peer-reviewed NEJM AI trial](https://pmc.ncbi.nlm.nih.gov/articles/PMC12768499)

### 5. Large-scale use of ambient scribes was reported in a health-system evaluation

- **Date:** March 31, 2025
- **Description:** *NEJM Catalyst* reported on a year-long evaluation of ambient AI scribe use within The Permanente Medical Group. From October 2023 through December 2024, 7,260 physicians used the technology across more than 2.5 million encounters. The authors estimated nearly 16,000 hours of documentation time saved.
- **Organizations involved:** The Permanente Medical Group and Kaiser Permanente; NEJM Catalyst.
- **Potential impact on healthcare:** This evaluation illustrates the operational scale to which clinical documentation tools had reached and provides a real-world complement to smaller workflow studies.
- **Evidence status and caveats:** This was an observational, health-system evaluation, not a randomized trial. The time saved is an estimate from one organization’s implementation and should not be treated as a universal effect.
- **Source URL:** [NEJM Catalyst article](https://www.ovid.com/journals/ncicd/pdf/10.1056/cat.25.0040~ambient-artificial-intelligence-scribes-learnings-after-1)

### 6. Generative AI was tested for patient-portal responses

- **Date:** July 30, 2025
- **Description:** A study in *Applied Clinical Informatics* assessed healthcare professionals’ ratings of synthetic GPT-generated responses to patient-portal messages compared with authentic provider-written responses. Evaluators rated generated responses more highly for empathy and readability; findings for relevance and medical accuracy were less definitive.
- **Organizations involved:** Researchers affiliated with the University of Pennsylvania, Vanderbilt University Medical Center, and other collaborators; *Applied Clinical Informatics*.
- **Potential impact on healthcare:** AI-generated drafts could help clinicians manage growing message volumes and improve the readability of patient communications.
- **Evidence status and caveats:** This was an evaluation of response quality, not evidence that autonomous AI replies improve patient outcomes or are safe for unsupervised use. Clinical accuracy and appropriate escalation remain essential.
- **Source URL:** [Peer-reviewed study](https://pmc.ncbi.nlm.nih.gov/articles/PMC12310298)

### 7. Google released open medical AI models for developers

- **Date:** July 9, 2025
- **Description:** Google Research announced MedGemma models for health-AI development, including multimodal models designed to work with medical images and text, as well as MedSigLIP, an image-and-text encoder. The models were presented as starting points for developers to evaluate and adapt to specific applications.
- **Organizations involved:** Google Research and Google DeepMind.
- **Potential impact on healthcare:** Open model weights and developer resources may lower the barrier to building and evaluating locally adapted medical AI applications, including tools that combine images with clinical text.
- **Evidence status and caveats:** The release was a research and development resource, not a clinically validated product or authorization for medical use. Google’s technical report cautions that benchmark results are only an initial step toward demonstrating real-world utility.
- **Source URLs:** [Google Research announcement](https://research.google/blog/medgemma-our-most-capable-open-models-for-health-ai-development); [MedGemma technical report](https://arxiv.org/html/2507.05201v3)

### 8. FDA published draft guidance on AI use in drug and biologic regulatory decisions

- **Date:** January 7, 2025
- **Description:** FDA issued draft guidance on using AI-generated information or data to support regulatory decisions about the safety, effectiveness, or quality of drugs and biological products. It proposed a risk-based approach to assessing model credibility for a defined context of use.
- **Organizations involved:** U.S. Food and Drug Administration; drug and biologic developers and other stakeholders.
- **Potential impact on healthcare:** A clearer evidence framework could help sponsors and regulators assess AI-derived information used in regulated development and review.
- **Evidence status and caveats:** The guidance was a draft and its focus was regulatory decision-making—not a general endorsement of AI drug discovery or a specific validation standard for every generative model.
- **Source URL:** [Federal Register notice](https://www.federalregister.gov/documents/2025/01/07/2024-31542/considerations-for-the-use-of-artificial-intelligence-to-support-regulatory-decision-making-for-drug)

### 9. FDA advanced a lifecycle approach for AI-enabled medical devices

- **Date:** January 7, 2025
- **Description:** FDA issued draft recommendations on lifecycle management and marketing submissions for AI-enabled device software functions. The agency asked for public comment on how the recommendations addressed emerging technologies, including generative AI, and on performance monitoring and user information.
- **Organizations involved:** U.S. Food and Drug Administration; medical-device developers and other stakeholders.
- **Potential impact on healthcare:** The proposed lifecycle emphasis could encourage developers to document validation, manage risks, and consider performance after deployment—not only during initial development.
- **Evidence status and caveats:** This was draft guidance for AI-enabled devices broadly, not GenAI-specific final regulation. The document did not settle all questions about generative or continuously changing systems.
- **Source URL:** [FDA announcement and draft-guidance information](https://www.fda.gov/news-events/press-announcements/fda-issues-comprehensive-draft-guidance-developers-artificial-intelligence-enabled-medical-devices)

### 10. FDA used generative AI internally and solicited advice on mental-health devices

- **Date:** May 8 and November 6, 2025
- **Description:** FDA announced that a generative AI pilot for scientific reviewers had concluded and directed agency centers toward deployment on a common secure platform. Later in the year, its Digital Health Advisory Committee discussed generative AI-enabled digital mental-health medical devices, including their benefits, risks, and possible risk mitigations.
- **Organizations involved:** U.S. Food and Drug Administration; FDA centers and its Digital Health Advisory Committee.
- **Potential impact on healthcare:** Internal tools may assist regulatory staff with tasks such as reading and summarizing documents. The mental-health-device discussion signaled that regulators were examining high-stakes patient-facing applications separately from general-purpose chatbots.
- **Evidence status and caveats:** FDA’s May announcement described an internal deployment plan but provided limited detail about the pilot’s methods and evaluation. The November committee meeting was advisory; it did not itself establish new binding rules.
- **Source URLs:** [FDA internal AI pilot announcement](https://www.fda.gov/news-events/press-announcements/fda-announces-completion-first-ai-assisted-scientific-review-pilot-and-aggressive-agency-wide-ai); [FDA mental-health-device meeting summary](https://www.fda.gov/media/190450/download)

### 11. Joint Commission and CHAI issued health-system governance guidance

- **Date:** September 17, 2025
- **Description:** The Joint Commission and the Coalition for Health AI (CHAI) released initial guidance on responsible AI use in healthcare. It emphasized organizational policies, local validation, monitoring, appropriate use, and integration with existing processes.
- **Organizations involved:** The Joint Commission and Coalition for Health AI.
- **Potential impact on healthcare:** The framework gives health systems a practical governance reference for procuring, validating, monitoring, and managing AI tools, including generative AI applications.
- **Evidence status and caveats:** This was non-binding organizational guidance, not a regulation or proof that any specific AI tool is safe.
- **Source URL:** [Joint Commission announcement](https://www.jointcommission.org/en-us/knowledge-library/news/2025-09-jc-and-chai-release-initial-guidance-to-support-responsible-ai-adoption)

### 12. EU AI Act obligations began applying in phases

- **Date:** February 2 and August 2, 2025
- **Description:** Under the EU AI Act, provisions on AI literacy and prohibited practices began applying on February 2, 2025; governance rules and obligations for general-purpose AI model providers began applying on August 2, 2025.
- **Organizations involved:** European Union institutions, national authorities, and providers and deployers of AI systems and general-purpose models.
- **Potential impact on healthcare:** The phased framework affects developers and deployers of general-purpose models that may underpin healthcare applications and adds obligations relevant to their governance and use.
- **Evidence status and caveats:** These dates mark phased application of a broad AI law, not a healthcare-specific clinical validation regime. The regulation’s high-risk provisions for medical devices had later application dates and were not fully in force in 2025.
- **Source URL:** [European Commission: AI Act application timeline](https://digital-strategy.ec.europa.eu/en/faqs/navigating-ai-act)

## 3. Emerging Trends

1. **From generic chatbots toward task-specific systems.** Examples included a therapy-oriented chatbot, models for medical images and text, and tools for drafting documentation or patient messages. Domain adaptation does not by itself establish clinical safety.
2. **More attention to human–AI workflow design.** The stronger clinical examples were generally assistive: AI drafted notes or reports for professionals to review, rather than making final decisions independently.
3. **A shift from benchmark scores toward workflow and clinical evidence.** Reader studies and randomized trials assessed time, report quality, or symptoms. However, evidence for long-term patient outcomes remained limited.
4. **Generative AI spread into operational work.** Documentation and portal-message handling were prominent targets because they consume clinician time. Reported efficiency gains varied by system and study design.
5. **Regulation and governance increasingly emphasized lifecycle oversight.** FDA draft guidance and public consultations, the Joint Commission–CHAI framework, and phased EU AI Act obligations reflected growing attention to validation, monitoring, transparency, and accountability.

## 4. Challenges and Risks

- **Hallucinations and omissions:** A fluent note, patient reply, or imaging report can still contain unsupported facts or miss important findings. Such errors may be difficult to spot when the output sounds authoritative.
- **Clinical evidence remains uneven:** A research trial or reader study is not equivalent to routine deployment. Many results are from controlled settings, limited populations, short follow-up, or single health systems. Demonstrated time savings do not automatically imply better health outcomes.
- **Human oversight and automation bias:** Review is a necessary safeguard, but it can be weakened if clinicians over-trust generated text or face pressure to approve drafts quickly. Responsibility for checking records and decisions remains a central implementation issue.
- **Bias and generalizability:** Models may perform differently across demographic groups, languages, specialties, and institutions. A 2025 study of four LLMs found relatively little difference in selected language and readability measures across race and ethnicity prompts, but those proxy measures do not establish equal clinical accuracy or equitable care.
- **Privacy and security:** Clinical conversations, portal messages, and records contain sensitive information. Organizations need clear controls over consent, data access, storage, retention, vendor use, and whether information may be used to train models. The Joint Commission–CHAI guidance explicitly calls for local governance and attention to privacy and transparency.
- **Accountability and regulatory fit:** Developers, health systems, clinicians, and regulators may have different responsibilities when AI outputs contribute to harm. In 2025, FDA’s relevant guidance was largely draft or exploratory, and EU AI Act duties were being phased in; important questions about post-deployment monitoring and model updates remained unresolved.
- **Mental-health-specific risks:** A chatbot may encounter crisis situations, self-harm, or other high-risk disclosures. The Therabot trial was a promising early study, but researchers and outside commentators stressed that it did not justify autonomous mental-health treatment or generalization to consumer chatbots.

## 5. Conclusion

In 2025, generative AI in healthcare made a meaningful transition from demonstrations toward more realistic evaluation. The clearest advances were the Therabot randomized trial, chest X-ray reporting studies, early clinical results for the AI-discovered rentosertib, and randomized evaluation of ambient scribes. Generative AI also continued to enter administrative workflows and developer toolkits.

The evidence does not support treating these systems as replacements for clinicians. Most results establish feasibility or benefit in narrow settings, not general clinical effectiveness. Future progress will depend on prospective evaluation in representative populations, transparent measurement of errors and outcomes, privacy-preserving deployment, and clear accountability for human and AI contributions.

## 6. References

- [Heinz et al., “Randomized Trial of a Generative AI Chatbot for Mental Health Treatment,” NEJM AI (March 27, 2025)](https://ai.nejm.org/doi/abs/10.1056/AIoa2400802)
- [Dartmouth, “First Therapy Chatbot Trial Yields Mental Health Benefits” (March 27, 2025)](https://home.dartmouth.edu/news/2025/03/first-therapy-chatbot-trial-yields-mental-health-benefits)
- [*Radiology*, “Value of Using a Generative AI Model in Chest Radiography Reporting” (March 11, 2025)](https://pubs.rsna.org/doi/10.1148/radiol.241646)
- [*Radiology*, “Generative AI Reporting of Chest Radiographs” (March 25, 2025)](https://pubs.rsna.org/doi/10.1148/radiol.250234)
- [Xu et al., “A generative AI-discovered TNIK inhibitor for idiopathic pulmonary fibrosis: a randomized phase 2a trial,” *Nature Medicine* (June 3, 2025)](https://www.nature.com/articles/s41591-025-03743-2)
- [Lukac et al., “Ambient AI Scribes in Clinical Practice: A Randomized Trial,” NEJM AI (November 26, 2025)](https://pmc.ncbi.nlm.nih.gov/articles/PMC12768499)
- [Tierney et al., “Ambient Artificial Intelligence Scribes: Learnings after 1 Year and over 2.5 Million Uses,” NEJM Catalyst (March 31, 2025)](https://www.ovid.com/journals/ncicd/pdf/10.1056/cat.25.0040~ambient-artificial-intelligence-scribes-learnings-after-1)
- [Kaur et al., “Automating Responses to Patient Portal Messages Using Generative AI,” *Applied Clinical Informatics* (July 30, 2025)](https://pmc.ncbi.nlm.nih.gov/articles/PMC12310298)
- [Google Research, “MedGemma: Our most capable open models for health AI development” (July 9, 2025)](https://research.google/blog/medgemma-our-most-capable-open-models-for-health-ai-development)
- [MedGemma Technical Report (2025)](https://arxiv.org/html/2507.05201v3)
- [FDA, draft guidance on AI for regulatory decision-making for drugs and biologics (Federal Register, January 7, 2025)](https://www.federalregister.gov/documents/2025/01/07/2024-31542/considerations-for-the-use-of-artificial-intelligence-to-support-regulatory-decision-making-for-drug)
- [FDA, draft guidance on AI-enabled device software functions (January 2025)](https://www.fda.gov/news-events/press-announcements/fda-issues-comprehensive-draft-guidance-developers-artificial-intelligence-enabled-medical-devices)
- [FDA, generative AI pilot for scientific review and agency-wide rollout announcement (May 2025)](https://www.fda.gov/news-events/press-announcements/fda-announces-completion-first-ai-assisted-scientific-review-pilot-and-aggressive-agency-wide-ai)
- [FDA Digital Health Advisory Committee meeting summary on generative AI-enabled mental-health devices (November 6, 2025)](https://www.fda.gov/media/190450/download)
- [Joint Commission and CHAI, responsible AI guidance announcement (September 17, 2025)](https://www.jointcommission.org/en-us/knowledge-library/news/2025-09-jc-and-chai-release-initial-guidance-to-support-responsible-ai-adoption)
- [European Commission, AI Act application timeline](https://digital-strategy.ec.europa.eu/en/faqs/navigating-ai-act)
- [Hanna et al., “Assessing Racial and Ethnic Bias in Text Generation by Large Language Models for Health Care–Related Tasks,” *Journal of Medical Internet Research* (March 13, 2025)](https://www.jmir.org/2025/1/e57257)
- [Tung et al., “Ethical and practical challenges of generative AI in healthcare and proposed solutions: a survey,” *Frontiers in Digital Health* (November 17, 2025)](https://pmc.ncbi.nlm.nih.gov/articles/PMC12665710)

### Exercise 2: AI Startups Landscape

Goal: Track 2025’s top emerging AI startups and their innovations.

- Create a prompt that instructs the agent to deliver a clean Markdown summary.
- Tip: Ask for company names, product highlights, and sources.

In [32]:
# Your prompt here
prompt_2 = """ 
You are a technology research analyst specializing in artificial
intelligence, startups, and emerging technologies.

Your task is to research the top emerging AI startups of 2026
and identify their most important innovations.

Use the Tavily web search tool to retrieve recent, reliable,
and verifiable information.

RESEARCH SCOPE:
- Focus on developments announced between January 1, 2026,
  and December 31, 2026.
- Since 2026 may still be ongoing, include only information
  published up to the current date.
- Identify 8–10 promising AI startups.
- Prioritize emerging companies rather than established
  technology giants.
- Cover multiple AI sectors, including:
  1. Generative AI and large language models
  2. AI agents and workflow automation
  3. Healthcare and biotechnology AI
  4. Robotics and physical AI
  5. AI infrastructure and developer tools
  6. Enterprise AI applications

SELECTION CRITERIA:
Evaluate startups based on:
- Technological innovation
- Product launches and technical breakthroughs in 2026
- Evidence of market adoption or commercial traction
- Funding and partnerships, where verifiable
- Potential impact on their respective industries

OUTPUT FORMAT:

Return a clean, professional Markdown report using
the following structure:

# Top Emerging AI Startups and Innovations in 2026

## 1. Executive Summary
Provide a brief overview of the AI startup landscape
and highlight the 3–5 most significant innovations.

## 2. Top Emerging AI Startups

For each startup, include:

### Company Name
- **Founded:** Year of establishment
- **Headquarters:** City and country
- **AI Sector:** Primary area of specialization
- **Key Innovation:** Description of the technology
- **Product Highlights:** Main products, features,
  capabilities, and advantages
- **2026 Developments:** Important launches, funding,
  partnerships, or technological achievements
- **Potential Impact:** How the innovation may transform
  its industry
- **Sources:** Direct URLs to reliable publications

## 3. Startup Comparison Table

Create a Markdown table with these columns:

| Startup | AI Sector | Key Product | Main Innovation | 2026 Milestone | Source |
|---------|-----------|-------------|-----------------|----------------|--------|

## 4. Emerging AI Trends in 2026

Identify 4–6 important trends across the selected startups.
Explain how their innovations reflect broader developments
in the AI industry.

## 5. Challenges and Opportunities

Discuss:
- Technical limitations
- Competition and scalability
- Regulatory and ethical concerns
- Commercialization opportunities

## 6. Conclusion

Summarize which startups appear most promising and why.
Distinguish documented achievements from predictions.

## 7. References

Provide a list of all sources in Markdown format,
including publication titles, dates, and working URLs.

IMPORTANT RULES:
- Use the Tavily search tool to gather evidence.
- Prioritize company announcements, reputable technology
  publications, investor reports, and credible research.
- Verify that each company actually exists.
- Do not invent funding figures, product launches,
  partnerships, dates, or sources.
- Clearly distinguish confirmed achievements from
  company claims and future expectations.
- If information cannot be verified, state this explicitly.
- Do not treat funding alone as proof of innovation.
- Avoid duplicate entries and outdated information.
- Use proper Markdown headings, bullets, and tables.
- Do not output raw JSON or Python dictionaries.
- Return only the final Markdown report.

"""

# Run it
response_2 = agent.invoke({
    "messages": [
        {"role": "user", "content": prompt_2}
    ]
})

In [33]:
last_message = response_2["messages"][-1]

final_answer_2 = last_message.content

if isinstance(final_answer_2, list):
    final_answer_2 = "\n".join(
        block["text"]
        for block in final_answer_2
        if isinstance(block, dict) and block.get("type") == "text"
    )

display(Markdown(final_answer_2))

# Top Emerging AI Startups and Innovations in 2026

*Research cutoff: October 21, 2026. This report includes developments announced from January 1 through the cutoff date; it does not make claims about the rest of the year. “Top” reflects a qualitative assessment of technical differentiation, product progress, evidence of adoption, and potential industry impact—not a funding-only ranking.*

## 1. Executive Summary

In 2026, a notable shift is underway from AI that primarily generates text or images to systems that can take actions: coordinating business workflows, interpreting clinical records, designing biological molecules, and controlling robots. Startups are also working on less visible but essential parts of the AI stack, especially serving models efficiently and giving agents reliable access to enterprise context.

Five innovations stand out:

1. **Robot foundation models are pairing general-purpose software with specialized hardware and training data.** Genesis AI’s announced GENE-26.5 and Generalist AI’s GEN-1 illustrate efforts to generalize manipulation across tasks and environments. Their published demonstrations and performance claims are promising, but broad, independent validation remains limited.
2. **AI agents are moving from single-task assistants toward coordinated, multi-step work.** Ema’s agents operate across enterprise applications, while Trace focuses on mapping company context and assigning work between people and agents.
3. **Clinical AI is targeting overlooked information in existing records.** Healthleap says its system flags patients who may need clinician review, rather than making diagnoses; it reports deployment in more than 50 hospitals.
4. **Generative AI is being tailored to scientific workflows.** Converge Bio combines models for biological sequences with antibody design, target discovery, and protein-manufacturing tasks. Its reported customer and program activity provides a commercial signal, not proof that AI-designed candidates will become medicines.
5. **Inference infrastructure and domain-specific applications are becoming strategic layers.** Inferact is commercializing the open-source vLLM serving project, while Legora applies AI to legal work. Both point to value beyond simply building a larger general-purpose model.

The companies below are a cross-sector shortlist, not a claim that each is a market leader. Where product performance, adoption, or clinical impact is based on company statements, that is identified explicitly.

## 2. Top Emerging AI Startups

### Runway

- **Founded:** 2018
- **Headquarters:** New York City, United States
- **AI Sector:** Generative AI, video, and world models
- **Key Innovation:** Runway’s core work combines generative video models with research into “world models”—systems intended to represent and simulate environments, rather than only generate a static response to a prompt. The world-model direction may eventually support interactive simulation, but that future potential should be distinguished from the company’s established media-generation products.
- **Product Highlights:** Runway’s products let creators generate and edit video using AI. Its models are used in media, entertainment, and advertising; TechCrunch also reported growing interest in gaming and robotics.
- **2026 Developments:** Runway announced a **$315 million Series E** in February, at a reported **$5.3 billion valuation**. The company said the funds would support pre-training its next generation of world models. TechCrunch reported that Runway had a partnership with Adobe and a compute deal with CoreWeave. Its first world model and Gen-4.5 pre-date the 2026 cutoff, so they are not presented here as 2026 launches.
- **Potential Impact:** If world models can reliably simulate how environments change, they could extend generative AI beyond content creation into simulation and planning. That remains an expectation, not a demonstrated commercial outcome.
- **Sources:**
  - [TechCrunch — “AI video startup Runway raises $315M at $5.3B valuation, eyes more capable world models” (February 10, 2026)](https://techcrunch.com/2026/02/10/ai-video-startup-runway-raises-315m-at-5-3b-valuation-eyes-more-capable-world-models)
  - [TechCrunch — “Runway started by helping filmmakers — now it wants to beat Google at AI” (May 15, 2026)](https://techcrunch.com/2026/05/15/runway-started-by-helping-filmmakers-now-it-wants-to-beat-google-at-ai)
  - [TechCrunch — company background and founding (June 29, 2023)](https://techcrunch.com/2023/06/29/runway-a-startup-building-generative-ai-for-content-creators-raises-141m)

### Nous Research

- **Founded:** 2023
- **Headquarters:** Austin, United States, according to public company profiles; the company’s precise operating footprint is not consistently reported.
- **AI Sector:** Open-weight language models and AI agents
- **Key Innovation:** Nous develops open-weight models and the Hermes agent. Hermes is an agent framework designed to retain memory, use tools, and perform multi-step tasks. The strategic distinction is an open-source route to agent deployment, rather than a closed assistant alone.
- **Product Highlights:** The Hermes Agent is designed to search the web, run code, handle scheduled tasks, and retain information across sessions. The company’s enterprise offering, Hermes for Businesses, is intended to let organizations deploy customized agents while keeping data private.
- **2026 Developments:** In October, Nous confirmed a **$90 million Series B** at a reported **$1.5 billion valuation** and launched Hermes for Businesses. TechCrunch reported that Hermes had been cloned more than 24 million times, citing a company estimate; clone counts indicate developer interest but do not establish sustained use or enterprise adoption.
- **Potential Impact:** If the agent proves reliable and governable in business settings, an open and customizable alternative could broaden enterprise access to agent technology. Security, model quality, and support obligations will matter as much as openness.
- **Sources:**
  - [TechCrunch — “Nous Research confirms it hit $1.5B valuation, launches AI agents for business users” (October 7, 2026)](https://techcrunch.com/2026/10/07/nous-research-confirms-it-hit-1-5b-valuation-launches-ai-agents-for-business-users)
  - [TechCrunch — “Hermes agent maker Nous Research in talks for new funding at $1.5B valuation” (July 13, 2026)](https://techcrunch.com/2026/07/13/hermes-agent-maker-nous-research-in-talks-for-new-funding-at-1-5b-valuation)
  - [Nous Research — Hermes Agent project](https://github.com/NousResearch/hermes-agent)

### Converge Bio

- **Founded:** 2024
- **Headquarters:** Boston, United States, and Tel Aviv, Israel
- **AI Sector:** Healthcare and biotechnology AI
- **Key Innovation:** Converge uses generative models trained on biological data—including DNA, RNA, and protein sequences—and combines them with predictive and physics-based components for scientific tasks. Its emphasis is on workflow-ready systems, rather than a single molecule-generation model.
- **Product Highlights:** The platform supports target discovery, antibody design, and protein-manufacturing optimization. The company says customers can use their own data to create private, fine-tuned models.
- **2026 Developments:** Converge announced a **$25 million Series A** in January, bringing reported funding to $30 million. The company said more than a dozen biotech and pharma customers were using its solutions and that it completed over 40 programs in the preceding year. These customer and performance figures are company-reported; they do not establish clinical efficacy or regulatory approval.
- **Potential Impact:** AI systems that help prioritize targets or design candidate molecules could reduce some early discovery work. Wet-lab experiments and later clinical studies remain essential to determine whether proposed candidates work safely in people.
- **Sources:**
  - [TechCrunch — “Converge Bio raises $25M, backed by Bessemer and execs from Meta, OpenAI, Wiz” (January 13, 2026)](https://techcrunch.com/2026/01/13/ai-drug-discovery-startup-converge-bio-pulls-in-25m-from-bessemer-and-execs-from-meta-openai-and-wiz)
  - [Converge Bio — funding and customer announcement (January 13, 2026)](https://www.prnewswire.com/news-releases/converge-bio-raises-25m-to-bring-generative-ai-drug-discovery-to-every-biotech-and-pharmaceutical-company-302659690.html)

### Healthleap

- **Founded:** 2022
- **Headquarters:** San Francisco, United States, according to funding coverage; the company was founded in South Africa.
- **AI Sector:** Healthcare AI and clinical workflow support
- **Key Innovation:** Healthleap analyzes structured electronic health record data and clinicians’ notes to identify possible signs of conditions that may have gone unnoticed. Its product flags cases for care teams; the company and reporting describe it as screening support, not an autonomous diagnostic system.
- **Product Highlights:** The platform initially focused on detecting disease-related malnutrition and has also developed programs for conditions including delirium and aspiration pneumonia. The company says it writes risk information into existing clinical workflows.
- **2026 Developments:** Healthleap announced **$38 million** in combined seed and Series A funding in October. TechCrunch reported that the platform was deployed in more than 50 hospitals. Fierce Healthcare reported a company-cited result from one hospital of 39% more malnutrition cases identified and treated; that specific result should not be treated as independently established across sites.
- **Potential Impact:** Finding care risks already documented in patient records could help clinicians prioritize review. The opportunity depends on accuracy, workflow fit, and evidence that alerts improve outcomes without adding excessive false positives or burden.
- **Sources:**
  - [TechCrunch — “Healthleap raises $38M for its AI that flags hospital patients who may need a closer look” (October 7, 2026)](https://techcrunch.com/2026/10/07/healthleap-raises-38m-for-its-ai-that-flags-hospital-patients-who-may-need-a-closer-look)
  - [Fierce Healthcare — “Healthleap banks $38M for AI screening platform that catches undiagnosed conditions” (October 8, 2026)](https://www.fiercehealthcare.com/finance/healthleap-banks-38m-funding-backed-sequoia-capital-first-round-capital-and-hummingbird)
  - [TechRepublic — funding and product overview (October 8, 2026)](https://www.techrepublic.com/article/news-healthleap-38m-ai-hospital-patient-screening)

### Genesis AI

- **Founded:** 2024
- **Headquarters:** Paris, France, and San Carlos, California, United States, according to its May 2026 announcement
- **AI Sector:** Robotics and physical AI
- **Key Innovation:** Genesis is pursuing a full-stack robotics approach: a foundation model, a dexterous robotic hand, and a data engine intended to transfer human-demonstrated skills to robots. The company’s premise is that better hardware and data collection can help address the shortage of varied, high-quality robot training data.
- **Product Highlights:** In May, the company announced GENE-26.5 alongside a robotic hand and data-collection system. It presented demonstrations of complex manipulation tasks. These demonstrations show what the company chose to showcase, not general-purpose performance across real-world settings.
- **2026 Developments:** Genesis unveiled GENE-26.5 on May 6. The announcement built on a **$105 million seed round raised in 2025**, which is relevant background but not a 2026 funding milestone. The “human-level” capability language in the launch materials is a company claim, not an independently verified benchmark result.
- **Potential Impact:** If the system can transfer skills efficiently across objects, tasks, and environments, it could make robots more flexible than task-specific automation. Reliability, safety, and the cost of deployment remain open questions.
- **Sources:**
  - [TechCrunch — “Khosla-backed robotics startup Genesis AI has gone full-stack, demo shows” (May 6, 2026)](https://techcrunch.com/2026/05/06/khosla-backed-robotics-startup-genesis-ai-has-gone-full-stack-demo-shows)
  - [Genesis AI — GENE-26.5 announcement (May 6, 2026)](https://www.prnewswire.com/news-releases/genesis-ai-unveils-gene-26-5--the-first-ai-brain-to-enable-robots-with-human-level-physical-manipulation-capabilities-302763638.html)

### Generalist AI

- **Founded:** 2024
- **Headquarters:** San Mateo, California, United States
- **AI Sector:** Robotics and embodied AI
- **Key Innovation:** Generalist’s GEN-1 is an embodied foundation model for controlling robots in physical tasks. The company says it uses large-scale human activity data for pre-training and can adapt to tasks with relatively little robot-specific data.
- **Product Highlights:** GEN-1 is designed to handle dexterous physical tasks and adapt to differing environments. Generalist’s April launch materials claimed 99% success on selected tasks and completion at roughly three times the speed of prior systems. Those figures are company-reported and should not be generalized to all tasks or deployments.
- **2026 Developments:** Generalist released GEN-1 in April. It also announced a **$400 million Series B in June**, reportedly valuing the company at $2 billion. Funding and valuation signal investor confidence, not proof of product performance or commercial deployment.
- **Potential Impact:** More transferable robot policies could reduce the need to program each task and robot separately. Commercial value will depend on whether performance holds in varied, safety-critical work environments.
- **Sources:**
  - [SiliconANGLE — “Generalist releases highly capable GEN-1 robotic intelligence AI foundation model” (April 6, 2026)](https://siliconangle.com/2026/04/06/generalist-releases-gen-1-highly-capable-robotic-intelligence-ai-foundation-model)
  - [Generalist AI — GEN-1 launch demonstration (April 2, 2026)](https://www.youtube.com/watch?v=SY2xyrmV44Y)
  - [Generalist AI — company site](https://generalistai.com/)

### Inferact

- **Founded:** 2025
- **Headquarters:** San Francisco, United States
- **AI Sector:** AI infrastructure and developer tools
- **Key Innovation:** Inferact is commercializing and developing **vLLM**, an open-source LLM inference engine created by members of its team. vLLM’s memory-management approach, including PagedAttention, aims to improve serving throughput and GPU memory use. The underlying project predates the 2026 company announcement; Inferact’s 2026 milestone is its formation and funding to develop the infrastructure further.
- **Product Highlights:** The open-source engine supports serving language models, with an aim to make inference faster and less expensive across different models and hardware. The project’s broad use is a useful adoption signal, but the startup’s future commercial success is not assured.
- **2026 Developments:** Inferact announced itself in January with a **$150 million seed round**, co-led by Andreessen Horowitz and Lightspeed Venture Partners, to support vLLM’s development and commercialization.
- **Potential Impact:** As model use scales, inference cost and latency become major constraints. Improving the serving layer could make AI products more affordable and reduce dependence on proprietary serving stacks.
- **Sources:**
  - [TechCrunch — “Inference startup Inferact lands $150M to commercialize vLLM” (January 24, 2026)](https://techcrunch.com/2026/01/24/inference-startup-inferact-lands-150m-to-commercialize-vllm)
  - [Lightspeed Venture Partners — Inferact company profile](https://lsvp.com/company/vllm-inferact)
  - [vLLM — open-source project](https://github.com/vllm-project/vllm)

### Ema

- **Founded:** 2023
- **Headquarters:** Mountain View, California, United States
- **AI Sector:** AI agents and enterprise workflow automation
- **Key Innovation:** Ema coordinates multiple AI agents to execute multi-step processes across a company’s existing applications. Rather than automating only one task, its approach is to connect tasks across business functions and route approvals where needed.
- **Product Highlights:** The company markets “AI Employees” for areas including HR, IT, and finance. Its agents are designed to work with existing enterprise software, check work, and request human approval where required.
- **2026 Developments:** Ema raised **$77 million in a Series B in September**, led by Creaegis, with Accel, Section 32, and Prosus participating. The company said its agents handle more than one million IT service tickets annually and one million calls across 15 languages; these are company-reported usage figures.
- **Potential Impact:** If agents can reliably complete processes across business systems, companies may automate work that currently spans multiple software tools and service teams. Permissioning, auditability, and recovery from errors will be crucial.
- **Sources:**
  - [TechCrunch — “Ema raises $77M as AI starts eating into enterprise software and services” (September 23, 2026)](https://techcrunch.com/2026/09/23/ema-raises-77m-as-ai-starts-eating-into-enterprise-software-and-services)
  - [Workday Ventures — Ema company profile](https://ventures.workday.com/en-us/partner-companies/ema.html)
  - [PYMNTS — Ema funding and company-reported usage (September 23, 2026)](https://www.pymnts.com/news/artificial-intelligence/2026/ema-secures-77-million-to-scale-ai-agents-across-enterprises)

### Trace

- **Founded:** Launched in 2025 as part of Y Combinator’s Summer 2025 cohort
- **Headquarters:** London, United Kingdom
- **AI Sector:** AI-agent orchestration and enterprise workflow automation
- **Key Innovation:** Trace builds a knowledge graph from a company’s existing tools and processes, then uses that context to coordinate tasks among AI agents and human workers. Its thesis is that agent adoption is limited not only by model capability, but also by lack of access to relevant organizational context.
- **Product Highlights:** Users can give the platform a high-level goal. Trace then proposes a workflow, assigns work to agents or people, and supplies agents with relevant context from connected company tools.
- **2026 Developments:** Trace announced a **$3 million seed round** in February. TechCrunch described the product and its early positioning but did not report large-scale adoption. Trace is an early-stage company with a technical thesis, not yet a demonstrated enterprise standard.
- **Potential Impact:** Context-aware orchestration could reduce the setup effort required to deploy agents inside companies. It will face competition from both enterprise software vendors and agent platforms that add similar integrations.
- **Sources:**
  - [TechCrunch — “Trace raises $3M to solve the AI agent adoption problem in enterprise” (February 26, 2026)](https://techcrunch.com/2026/02/26/trace-raises-3-million-to-solve-the-agent-adoption-problem)

### Legora

- **Founded:** 2023
- **Headquarters:** Stockholm, Sweden, with a major New York presence. TechCrunch described the company as headquartered in New York in March 2026; company and accelerator profiles also identify Stockholm as its base.
- **AI Sector:** Enterprise AI applications and legal technology
- **Key Innovation:** Legora combines legal research, document review, drafting, and workflow assistance in a platform designed for law firms and in-house legal teams. Its differentiator is an integrated legal work environment, rather than a general-purpose chatbot alone.
- **Product Highlights:** The platform supports review of legal documents, research, drafting, and collaborative legal workflows. Legal professionals remain responsible for checking output and legal advice.
- **2026 Developments:** Legora announced a **$550 million Series E at an $8.8 billion valuation on October 21**. TechCrunch reported that the company’s ARR had risen to more than $200 million, citing company disclosure. Earlier in 2026, it also expanded U.S. offices. The revenue and customer figures are reported company metrics, not independently audited here.
- **Potential Impact:** If accurate and well-integrated, legal AI could reduce time spent on document-heavy tasks and make legal services more efficient. Risks include inaccurate citations, confidentiality, jurisdiction-specific errors, and professional responsibility.
- **Sources:**
  - [TechCrunch — “Legal AI startup Legora raises $550M Series E at $8.8B valuation” (October 21, 2026)](https://techcrunch.com/2026/10/21/legal-ai-startup-legora-raises-550m-series-e-at-8-8b-valuation)
  - [TechCrunch — “Legora reaches $5.55 billion valuation as AI legal tech boom endures” (March 10, 2026)](https://techcrunch.com/2026/03/10/legora-reaches-5-55-billion-valuation-as-ai-legaltech-boom-endures)
  - [Y Combinator — Legora company profile](https://www.ycombinator.com/companies/legora)

## 3. Startup Comparison Table

| Startup | AI Sector | Key Product | Main Innovation | 2026 Milestone | Source |
|---------|-----------|-------------|-----------------|----------------|--------|
| Runway | Generative AI / world models | Video-generation tools and world-model research | Extending generative video toward interactive environment simulation | $315M Series E; funding directed toward next-generation world models | [TechCrunch, Feb. 10](https://techcrunch.com/2026/02/10/ai-video-startup-runway-raises-315m-at-5-3b-valuation-eyes-more-capable-world-models) |
| Nous Research | Open models / agents | Hermes Agent; Hermes for Businesses | Open-source agent with memory, tool use, and enterprise deployment ambitions | $90M Series B; business-agent product launched | [TechCrunch, Oct. 7](https://techcrunch.com/2026/10/07/nous-research-confirms-it-hit-1-5b-valuation-launches-ai-agents-for-business-users) |
| Converge Bio | Biotech AI | Generative biology platform | Combines generation, prediction, and physics-based methods for biological tasks | $25M Series A; company reported 12+ customers | [TechCrunch, Jan. 13](https://techcrunch.com/2026/01/13/ai-drug-discovery-startup-converge-bio-pulls-in-25m-from-bessemer-and-execs-from-meta-openai-and-wiz) |
| Healthleap | Healthcare AI | EHR-based patient risk screening | Extracts potential warning signs from records for clinician review | $38M combined funding; reported deployment in 50+ hospitals | [TechCrunch, Oct. 7](https://techcrunch.com/2026/10/07/healthleap-raises-38m-for-its-ai-that-flags-hospital-patients-who-may-need-a-closer-look) |
| Genesis AI | Robotics / physical AI | GENE-26.5 | Combines a robot foundation model with a dexterous hand and data engine | GENE-26.5 unveiled in May | [TechCrunch, May 6](https://techcrunch.com/2026/05/06/khosla-backed-robotics-startup-genesis-ai-has-gone-full-stack-demo-shows) |
| Generalist AI | Robotics / embodied AI | GEN-1 | Foundation model aimed at transferable manipulation skills | GEN-1 released in April; $400M Series B in June | [SiliconANGLE, Apr. 6](https://siliconangle.com/2026/04/06/generalist-releases-gen-1-highly-capable-robotic-intelligence-ai-foundation-model) |
| Inferact | AI infrastructure | vLLM inference engine | Optimizes model serving throughput and memory use | Launched with $150M seed funding | [TechCrunch, Jan. 24](https://techcrunch.com/2026/01/24/inference-startup-inferact-lands-150m-to-commercialize-vllm) |
| Ema | Enterprise agents | “AI Employees” | Coordinates agents across multi-step enterprise processes | $77M Series B; company reported large-scale ticket and call handling | [TechCrunch, Sept. 23](https://techcrunch.com/2026/09/23/ema-raises-77m-as-ai-starts-eating-into-enterprise-software-and-services) |
| Trace | Agent orchestration | Enterprise workflow and context graph | Supplies agents with organizational context and delegates work to people or agents | $3M seed round | [TechCrunch, Feb. 26](https://techcrunch.com/2026/02/26/trace-raises-3-million-to-solve-the-agent-adoption-problem) |
| Legora | Legal AI | Legal research and workflow platform | Integrates legal document review, research, and drafting | $550M Series E announced October 21 | [TechCrunch, Oct. 21](https://techcrunch.com/2026/10/21/legal-ai-startup-legora-raises-550m-series-e-at-8-8b-valuation) |

## 4. Emerging AI Trends in 2026

1. **AI systems are moving from outputs to actions.** Ema and Trace focus on multi-step workflows, while Legora packages AI around legal tasks. This shift raises the bar: systems must not only generate plausible content but also complete tasks correctly and safely.
2. **Context and integration are becoming competitive differentiators.** Trace’s knowledge graph and Ema’s connections to business applications reflect a broader effort to make agents useful within real organizations, where data is fragmented and permissions matter.
3. **Robotics is adopting foundation-model strategies.** Genesis AI and Generalist AI are trying to transfer skills across tasks rather than build a separate controller for each one. Their approaches emphasize data, embodiment, and hardware as well as model architecture.
4. **AI infrastructure is a major product category.** Inferact’s focus on inference reflects the cost and latency challenges of serving models at scale. The winners may be those that improve performance without tying customers to a single model or hardware vendor.
5. **Vertical AI is targeting high-value, specialized workflows.** Converge Bio, Healthleap, and Legora apply AI to scientific, clinical, and legal contexts. These areas have meaningful potential, but also demand domain expertise, validation, and strong accountability.
6. **Commercial signals are arriving faster than independent evidence.** Funding, deployments, customer counts, and company-reported performance claims can indicate market interest. They do not, by themselves, show durable retention, generalizable technical performance, or improved outcomes.

## 5. Challenges and Opportunities

- **Technical limitations:** Agents can make errors, misunderstand instructions, or fail when a workflow changes. In robotics, systems face the additional difficulty of variable objects, environments, and physical safety. Model demonstrations and selected benchmarks may not predict performance in everyday production settings.
- **Competition and scalability:** Startups compete with large AI labs, cloud providers, established software vendors, and other startups. A successful prototype may still be costly to operate, difficult to integrate, or hard to differentiate as general models improve.
- **Regulatory and ethical concerns:** Healthcare systems must address privacy, bias, clinical validation, and human oversight. Legal AI must protect confidential material and avoid misleading or incorrect legal output. Enterprise agents require access controls, audit logs, and clear responsibility for actions. Robotics adds workplace safety and labor impacts.
- **Commercialization opportunities:** The strongest opportunities appear where AI can fit into existing workflows and demonstrate measurable value—such as reducing administrative burden, improving infrastructure efficiency, or accelerating research iteration. In each sector, adoption will depend on trusted performance, integration costs, and evidence of return on investment.
- **Funding versus progress:** Large funding rounds can buy talent, compute, and time, but they are not a proxy for innovation or product-market fit. The more useful indicators are repeat usage, independent evaluations, successful deployments, and transparent reporting of limitations.

## 6. Conclusion

The most compelling near-term opportunities in this group are **Inferact** in inference infrastructure, **Ema** and **Trace** in enterprise agent orchestration, and **Legora** in a specialized legal workflow. Their products target concrete problems that organizations already spend money and time addressing. **Healthleap** is also notable for applying AI to existing clinical records, although clinical effectiveness must be demonstrated carefully.

For longer-term technical impact, **Genesis AI** and **Generalist AI** are pursuing the difficult goal of transferable physical intelligence. Their 2026 model announcements are significant research and product milestones, but company-reported demonstrations should not be confused with proven general-purpose robot performance. **Converge Bio** offers a potentially valuable scientific workflow platform, while **Runway** is investing in a longer-term transition from video generation toward world models.

These assessments distinguish **documented events**—such as launches, funding announcements, and reported deployments—from **future expectations** about industry transformation. The biggest unresolved question across the list is whether the current technical and commercial momentum can translate into reliable, repeatable value at scale.

## 7. References

- [“AI video startup Runway raises $315M at $5.3B valuation, eyes more capable world models” — TechCrunch, February 10, 2026](https://techcrunch.com/2026/02/10/ai-video-startup-runway-raises-315m-at-5-3b-valuation-eyes-more-capable-world-models)
- [“Runway started by helping filmmakers — now it wants to beat Google at AI” — TechCrunch, May 15, 2026](https://techcrunch.com/2026/05/15/runway-started-by-helping-filmmakers-now-it-wants-to-beat-google-at-ai)
- [“Nous Research confirms it hit $1.5B valuation, launches AI agents for business users” — TechCrunch, October 7, 2026](https://techcrunch.com/2026/10/07/nous-research-confirms-it-hit-1-5b-valuation-launches-ai-agents-for-business-users)
- [Hermes Agent project — Nous Research, GitHub](https://github.com/NousResearch/hermes-agent)
- [“Converge Bio raises $25M, backed by Bessemer and execs from Meta, OpenAI, Wiz” — TechCrunch, January 13, 2026](https://techcrunch.com/2026/01/13/ai-drug-discovery-startup-converge-bio-pulls-in-25m-from-bessemer-and-execs-from-meta-openai-and-wiz)
- [“Converge Bio raises $25M to bring generative AI drug discovery to every biotech and pharmaceutical company” — Converge Bio via PR Newswire, January 13, 2026](https://www.prnewswire.com/news-releases/converge-bio-raises-25m-to-bring-generative-ai-drug-discovery-to-every-biotech-and-pharmaceutical-company-302659690.html)
- [“Healthleap raises $38M for its AI that flags hospital patients who may need a closer look” — TechCrunch, October 7, 2026](https://techcrunch.com/2026/10/07/healthleap-raises-38m-for-its-ai-that-flags-hospital-patients-who-may-need-a-closer-look)
- [“Healthleap banks $38M for AI screening platform that catches undiagnosed conditions” — Fierce Healthcare, October 8, 2026](https://www.fiercehealthcare.com/finance/healthleap-banks-38m-funding-backed-sequoia-capital-first-round-capital-and-hummingbird)
- [“Khosla-backed robotics startup Genesis AI has gone full-stack, demo shows” — TechCrunch, May 6, 2026](https://techcrunch.com/2026/05/06/khosla-backed-robotics-startup-genesis-ai-has-gone-full-stack-demo-shows)
- [“Genesis AI Unveils GENE-26.5” — Genesis AI via PR Newswire, May 6, 2026](https://www.prnewswire.com/news-releases/genesis-ai-unveils-gene-26-5--the-first-ai-brain-to-enable-robots-with-human-level-physical-manipulation-capabilities-302763638.html)
- [“Generalist releases highly capable GEN-1 robotic intelligence AI foundation model” — SiliconANGLE, April 6, 2026](https://siliconangle.com/2026/04/06/generalist-releases-gen-1-highly-capable-robotic-intelligence-ai-foundation-model)
- [GEN-1 launch demonstration — Generalist AI, April 2, 2026](https://www.youtube.com/watch?v=SY2xyrmV44Y)
- [“Inference startup Inferact lands $150M to commercialize vLLM” — TechCrunch, January 24, 2026](https://techcrunch.com/2026/01/24/inference-startup-inferact-lands-150m-to-commercialize-vllm)
- [Inferact company profile — Lightspeed Venture Partners](https://lsvp.com/company/vllm-inferact)
- [“Ema raises $77M as AI starts eating into enterprise software and services” — TechCrunch, September 23, 2026](https://techcrunch.com/2026/09/23/ema-raises-77m-as-ai-starts-eating-into-enterprise-software-and-services)
- [Ema company profile — Workday Ventures](https://ventures.workday.com/en-us/partner-companies/ema.html)
- [“Trace raises $3M to solve the AI agent adoption problem in enterprise” — TechCrunch, February 26, 2026](https://techcrunch.com/2026/02/26/trace-raises-3-million-to-solve-the-agent-adoption-problem)
- [“Legal AI startup Legora raises $550M Series E at $8.8B valuation” — TechCrunch, October 21, 2026](https://techcrunch.com/2026/10/21/legal-ai-startup-legora-raises-550m-series-e-at-8-8b-valuation)
- [“Legora reaches $5.55 billion valuation as AI legal tech boom endures” — TechCrunch, March 10, 2026](https://techcrunch.com/2026/03/10/legora-reaches-5-55-billion-valuation-as-ai-legaltech-boom-endures)
- [Legora company profile — Y Combinator](https://www.ycombinator.com/companies/legora)

### Exercice 3: Compare Two Tech Products
- **Prompt idea:** Compare the key features, pricing, and reviews of OpenAI’s ChatGPT Team and Anthropic’s Claude Pro.
- Ensure the agent outputs a structured response in Markdown.




In [34]:
# Your prompt here
prompt_3 = """
You are an independent technology analyst specializing in
AI products, subscription plans, and software comparisons.

Your task is to conduct an objective, up-to-date comparison
between:

1. OpenAI ChatGPT Team (also known as ChatGPT Business)
2. Anthropic Claude Pro

Use the Tavily web search tool to retrieve current information
from official product websites, pricing pages, documentation,
and independent user reviews.

RESEARCH REQUIREMENTS:

1. FEATURES
Compare:
- Available AI models and capabilities
- Writing, reasoning, and coding assistance
- File uploads and document analysis
- Web search and research capabilities
- Integrations with external applications
- Collaboration and team features
- Usage limits and restrictions
- Data privacy and security protections
- Administrative controls

2. PRICING
For each plan, identify:
- Current monthly subscription price in USD
- Annual billing options, if available
- Minimum number of users or seats
- Total monthly cost for a team of five users
- Any relevant limitations or additional costs

Clearly distinguish per-user prices from total costs.
Include the date on which pricing was verified.

3. USER REVIEWS
Find independent reviews and user feedback from
reputable technology publications and review platforms.

Summarize:
- Common positive feedback
- Common complaints
- Ease of use
- Reliability and performance
- Customer support experiences
- Value for money

Distinguish verified user experiences from marketing claims.
Do not invent customer ratings or review scores.

4. COMPARATIVE ASSESSMENT
Evaluate each product for:
- Individual professionals
- Small business teams
- Researchers and students
- Software developers
- Organizations handling sensitive information

OUTPUT FORMAT:

Return a clean, professional Markdown report with
the following structure:

# ChatGPT Team vs Claude Pro: Comparison

## 1. Executive Summary
Briefly introduce the two products and explain the
most important differences.

## 2. Feature Comparison

Create a Markdown table:

| Feature | ChatGPT Team / Business | Claude Pro |
|---------|--------------------------|------------|

## 3. Pricing Comparison

Create a Markdown table:

| Pricing Category | ChatGPT Team / Business | Claude Pro |
|------------------|--------------------------|------------|
| Monthly Price | ... | ... |
| Annual Billing | ... | ... |
| Minimum Seats | ... | ... |
| Cost for 5 Users | ... | ... |

Explain any relevant pricing differences.

## 4. User Reviews and Feedback

### ChatGPT Team / Business
- Main advantages
- Common complaints
- Overall user sentiment

### Claude Pro
- Main advantages
- Common complaints
- Overall user sentiment

## 5. Strengths and Weaknesses

Summarize the main pros and cons of each subscription.

## 6. Recommendations

Explain which product is better suited to different
types of users and use cases.

## 7. Conclusion

Provide a balanced final assessment based on the
available evidence.

## 8. Sources and References

Include direct Markdown hyperlinks to all major sources,
including official pricing pages and independent reviews.

IMPORTANT RULES:
- Search the web before producing the comparison.
- Use the most recently verified information available.
- Clearly state the research date.
- Prefer official sources for prices, product features,
  security policies, and subscription terms.
- Use independent sources for reviews and opinions.
- Distinguish facts from opinions and interpretations.
- Do not invent prices, review ratings, or features.
- Explain when the two plans are not directly comparable.
- Identify unavailable or conflicting information.
- Use clear Markdown headings, tables, and bullet points.
- Do not return JSON or raw search results.
- Output only the final Markdown report.
"""

# Run it
response_3 = agent.invoke({
    "messages": [
        {"role": "user", "content": prompt_3}
    ]
})

In [35]:
final_answer_3 = response_3["messages"][-1].content

if isinstance(final_answer_3, list):
    final_answer_3 = "\n".join(
        block["text"]
        for block in final_answer_3
        if isinstance(block, dict) and block.get("type") == "text"
    )

display(Markdown(final_answer_3))

# ChatGPT Team vs Claude Pro: Comparison

**Research date: 17 August 2026.** Prices and plan details below were checked against the vendors’ current pricing and help pages. Features, regional availability, and usage limits can change; check the linked pages before purchasing.

## 1. Executive Summary

ChatGPT Team was renamed **ChatGPT Business**; OpenAI says the name change did not alter the plan’s features or terms. It is a managed workspace subscription for teams, with centralized billing, administrative controls, business-data protections, and access to ChatGPT tools and integrations.

**Claude Pro is an individual subscription**, not Anthropic’s team plan. It provides higher usage than Claude Free and access to Claude features such as Projects, web search, and Claude Code, but does not provide the same team workspace, centralized administration, or organization-wide controls as ChatGPT Business.

For five people, five Claude Pro subscriptions cost less than five standard ChatGPT Business seats. The plans are not like-for-like, however: Business adds shared administration and business-oriented data protections, while Pro is designed for individual users.

## 2. Feature Comparison

| Feature | ChatGPT Team / Business | Claude Pro |
|---------|--------------------------|------------|
| **Plan type** | Shared business workspace with individual seats, centralized billing, and admin controls. | Individual consumer subscription. It does not provide the team workspace or centralized administration of a business plan. |
| **Available models and capabilities** | Provides access to the models and tools available to Business users. The model menu and limits can change; access to some advanced capabilities is usage-limited or credit-based. | Provides access to more Claude models than Free, subject to Anthropic’s current model availability and Pro usage limits. |
| **Writing and reasoning** | General-purpose writing, reasoning, analysis, and work assistance. Includes tools such as Projects and custom GPTs. | General-purpose writing, analysis, and reasoning. Projects provide a way to organize chats and reference project files. |
| **Coding** | Includes coding assistance and access to Codex features, subject to plan limits. | Includes Claude Code; its usage draws on the same plan allowance as Claude chat. |
| **File uploads and document analysis** | Supports document uploads, data analysis, and project files. Current help documentation lists up to 40 files per project for Business; file size and other upload restrictions apply. | Supports file uploads in chats and Projects, including common document formats and images. Anthropic documents limits such as up to 20 files per chat and 500 MB per file; other format, page, and context limits apply. |
| **Web search and research** | Provides web search and Deep Research; Business can also use supported connected sources in research workflows. | Provides web search and web fetch. Anthropic’s tools and help materials describe research and source retrieval, subject to usage limits and regional availability. |
| **External integrations** | Business pricing lists connections to tools such as Google Workspace, Slack, GitHub, and Microsoft 365, with administrators able to manage access. | Pro includes some app and browser integrations, including Claude in Chrome and Microsoft 365 on the current pricing page. More extensive workplace connectors and organization-wide search are described for Anthropic’s Team plan, not Pro. |
| **Collaboration and team features** | Shared workspace, shared projects, centralized billing, workspace-level controls, and member management. | Individual Projects and personal integrations, but no comparable shared team workspace or centrally managed seat pool on Pro. |
| **Usage limits and restrictions** | Standard seats include defined access to models and tools; advanced usage may have limits, and OpenAI offers workspace credits to extend some usage. Limits differ by feature and may change. | Pro is not unlimited. Usage varies with model, prompt and file size, conversation length, and tools used. Anthropic documents session-based limits that reset every five hours and weekly limits; additional usage may be available through credits. |
| **Privacy and model training** | OpenAI says it does not train on Business workspace data by default. Business data is encrypted in transit and at rest. | Pro is a consumer plan. Anthropic says use of consumer chats for model improvement is controlled through the user’s privacy settings; verify the current setting before using sensitive material. |
| **Administrative controls and security** | Includes workspace administration, member and role management, billing controls, and SAML SSO. It is a business plan, but does not include every Enterprise control. | Pro has personal account settings, not organization-level administration. Anthropic describes stronger centralized controls for its separate Team and Enterprise plans. |

**Comparability note:** ChatGPT Business is a business workspace plan; Claude Pro is for individuals. Anthropic’s Team plan—not Pro—is the closer plan-type comparison to ChatGPT Business.

## 3. Pricing Comparison

| Pricing Category | ChatGPT Team / Business | Claude Pro |
|------------------|--------------------------|------------|
| **Monthly Price** | **$25 per standard seat per month**, billed monthly. | **$20 per user per month**, billed monthly. |
| **Annual Billing** | **$20 per standard seat per month**, billed annually; $240 per seat per year. | **$200 per user per year**, equivalent to about **$16.67 per month**. |
| **Minimum Seats** | At least **2 standard seats**. | No team-seat minimum for individual Pro subscriptions; each person subscribes separately. |
| **Cost for 5 Users** | Standard seats: **$125/month** on monthly billing, or **$1,200/year** billed annually (equivalent to $100/month). | Five Pro subscriptions: **$100/month** on monthly billing, or **$1,000/year** billed annually (equivalent to about $83.33/month). |

The five-user totals assume US pricing, standard ChatGPT Business seats, no taxes, and no additional usage credits. Regional prices and taxes may differ. Claude Pro annual pricing requires an upfront yearly payment. ChatGPT Business offers monthly and annual billing. Extra usage, where applicable, can add cost; **Claude Pro does not include Claude API usage**, which is billed separately.

## 4. User Reviews and Feedback

The available independent reviews are not a controlled, plan-to-plan comparison. Reviews of ChatGPT often concern the general ChatGPT product rather than Business specifically; Claude Pro-specific feedback can also be mixed with reviews of Claude more broadly. Treat review themes as user reports, not guaranteed outcomes.

### ChatGPT Team / Business

- **Main advantages:** Reviews of ChatGPT commonly praise ease of use, speed to get started, broad usefulness, and help with writing, ideas, and everyday work. Verified-user feedback on review platforms also describes productivity benefits and a relatively low learning curve.
- **Common complaints:** Users report inaccurate or inconsistent answers, a need to check important outputs, and responses that can be too long or repetitive. These are general ChatGPT complaints, not necessarily specific to Business.
- **Ease of use:** Commonly described as accessible, with a familiar conversational interface.
- **Reliability and performance:** User feedback is mixed: reviewers praise its usefulness and speed for many tasks, while also noting occasional errors or variability. AI output should be checked, especially for consequential work.
- **Customer support:** Public user-review evidence does not establish a consistent Business-specific support experience. Do not infer an enterprise support guarantee from the consumer reviews.
- **Overall user sentiment:** Broadly positive about versatility and productivity, tempered by the need to verify outputs. The evidence available is not sufficient to draw a separate, robust sentiment conclusion for the Business plan alone.

### Claude Pro

- **Main advantages:** Independent reviews and user feedback often praise Claude’s writing, ability to handle long documents or context, and coding assistance. A *Tom’s Guide* hands-on comparison of consumer subscriptions favored Claude Pro for its tested coding and reasoning tasks, while noting areas where ChatGPT was stronger.
- **Common complaints:** Usage limits are a recurring concern. Anthropic’s own documentation confirms that Pro usage is limited and depends on factors such as conversation length, files, tools, and model choice. Users also report that limits can interrupt longer work.
- **Ease of use:** Reviewers often describe Claude as straightforward to use; its individual subscription and Projects are oriented toward personal workflows rather than team administration.
- **Reliability and performance:** User experiences vary. Reviewers praise some outputs and coding workflows, but no single review establishes consistent performance across all tasks or users.
- **Customer support:** Available public feedback is anecdotal and does not establish a dependable Pro-specific support pattern.
- **Overall user sentiment:** Positive feedback tends to emphasize writing, long-context work, and coding; the clearest recurring caveat is limited usage. Platform reviews should be consulted directly because feedback and sample sizes change.

## 5. Strengths and Weaknesses

### ChatGPT Business

**Strengths**
- Designed for shared use by a small team, with centralized billing and workspace administration.
- Business-data protections include no training on workspace data by default.
- Offers a broad set of ChatGPT tools and integrations for organizational workflows.
- Five standard seats can be managed together rather than as five unrelated individual subscriptions.

**Weaknesses**
- Costs more than five Claude Pro subscriptions at the listed US prices.
- Some advanced usage can be limited or require additional workspace credits.
- Business is not equivalent to Enterprise: organizations needing more advanced identity, compliance, or contractual controls should check the Enterprise terms.

### Claude Pro

**Strengths**
- Lower listed cost for five individual subscriptions, particularly with annual billing.
- Includes Claude Code and Projects, and supports file-based work and web search.
- Independent hands-on comparisons and user feedback often praise its writing, reasoning, and coding assistance.

**Weaknesses**
- It is not an organizational plan: no equivalent team workspace, centralized seat administration, or shared billing for a five-person group.
- Usage limits can interrupt intensive work and are affected by conversation and tool use.
- Consumer-plan privacy settings differ from business-plan protections; it is not the straightforward choice for centrally governed sensitive data.

## 6. Recommendations

- **Individual professionals:** Claude Pro is the lower-cost individual subscription, especially on annual billing. ChatGPT Business is generally excessive for a single person unless that person needs a managed business workspace or business-oriented data handling.
- **Small business teams:** ChatGPT Business is the more direct fit when shared administration, centralized billing, and workspace controls matter. Five Claude Pro subscriptions cost less, but do not provide equivalent team management.
- **Researchers and students:** Either can assist with documents and web-based research. Compare the available research features, file needs, and usage caps for your workflow. For institutional or sensitive research, follow your institution’s data rules rather than relying only on a consumer plan.
- **Software developers:** Both offer coding help. Claude Pro includes Claude Code, while ChatGPT Business includes coding features and Codex access. Compare actual usage limits and your preferred workflow; API usage is separate from Claude Pro.
- **Organizations handling sensitive information:** ChatGPT Business has an explicit business workspace and states that workspace data is not used for model training by default. Claude Pro is a consumer plan with user-controlled data settings; organizations should evaluate Anthropic’s Team or Enterprise plans instead if they need centrally governed access and business terms. Neither subscription should be assumed to satisfy a particular compliance requirement without checking the applicable vendor terms.

## 7. Conclusion

These plans serve different needs. **ChatGPT Business is the better fit when a team needs a shared workspace, centralized billing, and administrative controls. Claude Pro is a lower-cost individual subscription for people who want Claude’s personal productivity and coding tools.**

For five users, Claude Pro costs less on the published US prices, but it is not a like-for-like alternative to Business. The decision should turn on whether the priority is individual cost and usage—or team administration and business data protections—as well as on each group’s tolerance for usage limits.

## 8. Sources and References

### Official product, pricing, and documentation

- [OpenAI Business pricing](https://openai.com/business/chatgpt-pricing)
- [ChatGPT Business overview and pricing](https://help.openai.com/en/articles/8792828-chatgpt-business-overview)
- [Managing billing and seats in ChatGPT Business](https://help.openai.com/en/articles/8792536-managing-billing-and-seats-in-chatgpt-business)
- [ChatGPT Business rename FAQ](https://help.openai.com/en/articles/12111915-chatgpt-business-rename-faq)
- [ChatGPT Business models and limits](https://help.openai.com/en/articles/12003714-chatgpt-business-models-and-limits)
- [Projects in ChatGPT: file limits and collaboration](https://help.openai.com/en/articles/10169521-projects-in-chatgpt)
- [ChatGPT file upload limits](https://help.openai.com/en/articles/8555545-uploading-files-and-audio-to-chatgpt)
- [OpenAI business data privacy](https://openai.com/business/data-privacy)
- [Setting up SSO for ChatGPT Business](https://help.openai.com/en/articles/11489188-setting-up-single-sign-on-sso-for-chatgpt-business)
- [Claude pricing](https://claude.com/pricing)
- [What is Claude Pro?](https://support.anthropic.com/en/articles/8325606-what-is-claude-pro)
- [Claude Pro usage limits and best practices](https://support.anthropic.com/en/articles/9797557-usage-limit-best-practices)
- [Claude file upload documentation](https://support.anthropic.com/en/articles/8241126-what-kinds-of-documents-can-i-upload-to-claude-ai)
- [Enabling and using web search in Claude](https://support.anthropic.com/en/articles/10684626-enabling-and-using-web-search)
- [Anthropic: Is my data used for model training?](https://privacy.anthropic.com/en/articles/7996868-is-my-data-used-for-model-training)
- [Claude Pro annual billing instructions](https://support.anthropic.com/en/articles/10185996-how-to-change-your-pro-plan-from-monthly-to-annual-billing)

### Independent reviews and user feedback

- [Tom’s Guide: Claude Pro vs ChatGPT Plus hands-on comparison](https://www.tomsguide.com/ai/claude-pro-vs-chatgpt-plus-i-tested-both-subscriptions-to-see-which-ones-actually-worth-usd20) — editorial test of individual plans, not ChatGPT Business.
- [TechRadar: Report on Claude usage limits](https://www.techradar.com/ai-platforms-assistants/claude/claude-is-limiting-usage-more-aggressively-during-peak-hours-heres-what-changed) — reporting on usage-limit concerns, not a user-rating survey.
- [G2 ChatGPT reviews](https://www.g2.com/products/chatgpt/reviews?qs=pros-and-cons) — user feedback on ChatGPT generally, not exclusively Business.
- [Software Advice ChatGPT reviews](https://www.softwareadvice.com/artificial-intelligence/chatgpt-profile/reviews) — user feedback including verified-review disclosures; not exclusively Business.
- [G2 Claude reviews](https://www.g2.com/products/claude-2026-10-27/reviews) — user feedback on Claude; check review dates and plan details.
- [Capterra Claude reviews](https://www.capterra.com/p/10011218/Claude/reviews) — user feedback on Claude; reviews may not all concern Pro specifically.

## Bonus Task: Propose and Implement Your Own Use Case

As a final challenge, think of a real-world scenario where an AI agent could provide value using web search or external tools.


In [36]:
prompt_4 = """
You are a German legal research assistant specializing in
German civil law, criminal law, and legislative developments.

Your task is to identify and analyze the most significant
changes to the German Civil Code (Bürgerliches Gesetzbuch,
BGB) and German Criminal Code (Strafgesetzbuch, StGB)
between January 1, 2022, and November 2, 2026.

Use the Tavily web search tool to research legislative
amendments and verify your findings against authoritative
German legal sources.

RESEARCH SOURCES:

Prioritize:
1. Bundesgesetzblatt (recht.bund.de)
2. Gesetze im Internet (gesetze-im-internet.de)
3. Deutscher Bundestag (bundestag.de)
4. Bundesministerium der Justiz (bmj.de)
5. Bundesgerichtshof (bundesgerichtshof.de)

Use reputable legal commentary only as a secondary source.

RESEARCH SCOPE:

PART A: GERMAN CIVIL LAW (BGB)

Identify important amendments in:
- Contract law and consumer protection
- Sales law and digital products
- Law of obligations and damages
- Property law
- Family law
- Inheritance law
- Legal capacity, guardianship, and representation
- Limitation periods, where applicable

PART B: GERMAN CRIMINAL LAW (StGB)

Identify important amendments in:
- General principles of criminal liability
- Sentencing and criminal sanctions
- Offences against life and physical integrity
- Sexual offences
- Economic and financial crime
- Cybercrime
- Hate crime and offences involving digital communication
- Drug-related offences, where the StGB was amended

For both codes, distinguish substantive amendments
from purely editorial or technical changes.

FOR EACH IMPORTANT AMENDMENT, PROVIDE:

1. Name of the amending legislation (German title)
2. Relevant BGB or StGB sections (§§)
3. Date of enactment and date of entry into force
4. Previous legal position
5. New legal position
6. Purpose of the reform
7. Practical implications, including examples
8. Official source URL

Where possible, explain the change through a short
hypothetical legal case.

OUTPUT FORMAT:

Return a well-structured Markdown report in English.
Retain German statutory names and legal terminology.

# Major Changes to German Civil and Criminal Law (2022–2026)

## 1. Executive Summary
Summarize the most important legislative developments.

## 2. Changes to the German Civil Code (BGB)

For each significant reform, provide:

### Reform Name
- **Relevant provisions:**
- **Amending legislation:**
- **Entry into force:**
- **Previous law:**
- **Current law:**
- **Practical significance:**
- **Example:**
- **Official source:**

Include a comparison table:

| BGB Section | Previous Law | New Law | Effective Date |
|-------------|--------------|---------|----------------|

## 3. Changes to the German Criminal Code (StGB)

Use the same structure for criminal law reforms.

Include a comparison table:

| StGB Section | Previous Law | New Law | Effective Date |
|--------------|--------------|---------|----------------|

## 4. Timeline of Legislative Changes

Create a chronological overview grouped by year:
- 2022
- 2023
- 2024
- 2025
- 2026 (up to November 2)

## 5. Practical Implications

Explain how the reforms affect:
- Private individuals
- Businesses
- Lawyers and legal practitioners
- Courts and criminal justice authorities

## 6. Most Relevant Changes for German Legal Examinations

Identify reforms particularly relevant to the
First and Second German State Examinations.

Explain their relevance to:
- Civil law cases and judicial decisions
- Criminal law cases
- Procedural or practical legal questions

Clearly distinguish confirmed examination requirements
from your own assessment of likely relevance.

## 7. Conclusion

Summarize the main developments in German law since 2022.

## 8. Official Sources and References

Provide direct Markdown hyperlinks to the legislation
and official legal materials.

CRITICAL ACCURACY REQUIREMENTS:

- Verify all statutory amendments using official sources.
- Do not invent legislation, provisions, or effective dates.
- Cite the specific amending statute for every change.
- Where possible, identify the relevant Bundesgesetzblatt
  citation (BGBl. year and number).
- Distinguish publication, enactment, and entry into force.
- Identify transitional provisions when relevant.
- Do not confuse changes to the BGB or StGB with changes
  to other statutes, including the ZPO, StPO, or KCanG.
- Do not confuse changes in case law with statutory amendments.
- Do not describe government proposals as enacted law.
- If the previous legal wording cannot be verified,
  explicitly state this instead of reconstructing it.
- If evidence is insufficient, explain the uncertainty.
- Do not claim the report is exhaustive unless every
  amendment has been systematically verified.
- Use proper Markdown headings, tables, and bullet points.
- Return only the final Markdown report.
"""

response_4 = agent.invoke({
    "messages": [
        {"role": "user", "content": prompt_4}
    ]
})

In [37]:
from IPython.display import Markdown, display

final_answer_4 = response_4["messages"][-1].content

if isinstance(final_answer_4, list):
    final_answer_4 = "\n".join(
        block["text"]
        for block in final_answer_4
        if isinstance(block, dict) and block.get("type") == "text"
    )

display(Markdown(final_answer_4))

# Major Changes to German Civil and Criminal Law (2022–2026)

> **Scope and date note.** This report selects reforms that materially changed the text or operation of the BGB or StGB; it is not a clause-by-clause inventory of every amendment. Some major reforms were enacted in 2021 but took effect during the requested period, and are included on that basis. **“Enacted” below means the date of enactment shown in the statute; publication is separately identified where verified.** The period ends on **2 November 2026**. The fact that a provision has been enacted does not necessarily mean it was already in force by that date.  
>
> The official consolidated texts at *Gesetze im Internet* are useful for checking current wording, but the site itself explains that the authoritative promulgated version is the **Bundesgesetzblatt (BGBl.)**. ([Official notice](https://www.gesetze-im-internet.de/))

## 1. Executive Summary

The most consequential BGB developments in the period are:

- **2022 digital-contract and sales-law reforms:** new consumer-contract rules for digital products and significant changes to consumer sales remedies, including for goods with digital elements.
- **2023 guardianship and adult-representation reform:** a comprehensive reorganisation of Betreuung and Vormundschaft, a stronger statutory priority for the wishes of the person receiving support, and a limited emergency representation right for spouses in health matters.
- **2024 partnership-law modernisation (MoPeG):** a fundamental statutory recasting of the **Gesellschaft bürgerlichen Rechts (GbR)**, including its legal capacity, registration and property-holding framework.
- **2025 non-residential leasing form reform:** commercial and other non-residential leases moved from a written-form to a text-form standard for the effect of § 550 BGB.
- **2026 consumer and parentage changes:** an online withdrawal function for distance contracts and new consumer information duties; a separate reform replaced the former BGB procedure addressing potentially abusive acknowledgments of paternity.

The most consequential StGB developments are:

- **2022 repeal of § 219a StGB**, the offence of advertising abortion services, and addition of a new **§ 130(5) StGB** offence concerning certain public statements about international crimes.
- **2023 sanctions reform:** notably, one day of *Ersatzfreiheitsstrafe* is now generally based on two day-fine units rather than one, alongside changes to sentencing and treatment orders.
- **2024 adjustment of § 184b StGB:** reduced minimum penalties for specified child-pornography offences, restoring their classification as *Vergehen* rather than *Verbrechen*.
- **2024 introduction of § 108f StGB:** criminal liability for specified forms of paid, impermissible interest representation by covered office-holders.
- **2026 terrorism and espionage reform:** a substantial revision of the terrorism-preparation offence and related provisions, including the penalty framework for secret-service agent activity.

Many developments in cybercrime, sexual offences and drug law during this period concerned **other statutes**, not the StGB. They are not treated here as amendments to the StGB.

---

## 2. Changes to the German Civil Code (BGB)

### 2.1 Digital products and consumer contracts

#### Reform: Contracts for digital products

- **Relevant provisions:** §§ 327–327u BGB; related provisions include §§ 312, 327a and 475a.
- **Amending legislation:** *Gesetz zur Umsetzung der Richtlinie über bestimmte vertragsrechtliche Aspekte der Bereitstellung digitaler Inhalte und digitaler Dienstleistungen*; enacted 25 June 2021, published **BGBl. 2021 I, p. 2123**; effective **1 January 2022**.
- **Previous law:** The BGB did not have this general, harmonised consumer-contract regime specifically for the supply of digital content and digital services. The consumer’s remedies depended more heavily on the underlying contract type and applicable general rules.
- **Current law:** §§ 327 ff. provide a dedicated framework for consumer contracts for digital content and digital services. It regulates, among other matters, supply, conformity, updates, remedies and contract termination. The framework can apply whether the supply is paid for in money or, in defined circumstances, provided in return for personal data.
- **Purpose:** To implement Directive (EU) 2019/770 and provide consistent consumer protection as digital products became a routine part of consumer transactions.
- **Practical significance:** Consumers have a statutory framework for defective or undelivered digital products, rather than having to rely solely on rules designed for conventional goods or services.
- **Example:** A consumer pays for a streaming service that repeatedly fails to provide the contracted catalogue. The consumer can assess supply and conformity remedies under the digital-products provisions.
- **Official source:** [BMJ/BGBl law text](https://www.bmjv.de/SharedDocs/Downloads/DE/Gesetzgebung/BGBl/Bgbl_Digitale_Inhalte.pdf?__blob=publicationFile&v=3); [BMJ overview of the 2022 BGB update](https://www.bmj.de/SharedDocs/Pressemitteilungen/DE/2021/1230_Kaufrecht.html).

#### Reform: Consumer sales law, including goods with digital elements

- **Relevant provisions:** In particular §§ 434, 437, 439, 441, 475b–475e BGB.
- **Amending legislation:** *Gesetz zur Regelung des Verkaufs von Sachen mit digitalen Elementen und anderer Aspekte des Kaufvertrags*; enacted 25 June 2021, published in **BGBl. 2021 I No. 37**; effective **1 January 2022**.
- **Previous law:** Consumer sales law did not contain the current tailored regime for goods whose functionality depends on digital content or digital services. The period during which a defect was presumed to have existed at delivery was shorter.
- **Current law:** The BGB now sets specific conformity and update duties for consumer goods with digital elements. The seller may be responsible for necessary updates over the relevant period. The period for the consumer-favourable presumption that a defect existed at delivery was extended to one year for consumer sales.
- **Purpose:** To implement Directive (EU) 2019/771 and respond to products whose functionality depends on software or connected services.
- **Practical significance:** A seller may have continuing update obligations after delivery; a consumer’s statutory remedies may apply where a device becomes defective or insecure because required updates are not provided.
- **Example:** A consumer buys a connected thermostat that stops communicating with its app because the seller fails to provide a required update during the relevant period. The digital-elements sales rules may be material to whether the goods conform to the contract.
- **Official source:** [BMJ law page](https://www.bmjv.de/SharedDocs/Downloads/DE/Gesetzgebung/BGBl/Bgbl_Regelung_des_Verkaufs_von_Sachen_mit_digitalen_Elementen.html); [BMJ summary of the sales-law changes](https://www.bmj.de/SharedDocs/Meldungen/DE/2021/1230_Neuregelungen_im_Kaufrecht.html).

### 2.2 Consumer protection and ongoing contracts

#### Reform: Fair consumer contracts

- **Relevant provisions:** §§ 309 no. 9 and 312k BGB; associated transition rules in the EGBGB.
- **Amending legislation:** *Gesetz für faire Verbraucherverträge*; enacted 10 August 2021, **BGBl. 2021 I, p. 3433**. Different provisions took effect on different dates: relevant AGB rules on **1 March 2022** and the online cancellation-button duty on **1 July 2022**.
- **Previous law:** For covered consumer contracts, AGB could provide for automatic renewal for up to a year and notice periods of up to three months. Covered online contracts did not yet have the statutory cancellation-button duty.
- **Current law:** For covered contracts made from 1 March 2022, automatic renewal clauses must generally permit termination of the renewed contract at any time on no more than one month’s notice; notice periods for termination at the end of the initial term are also limited. Under § 312k, covered online businesses must provide an accessible online means of cancellation.
- **Purpose:** To prevent consumers from being locked into long renewals and to make cancellation of qualifying online contracts practical.
- **Practical significance:** Businesses need to review renewal clauses and ensure the cancellation function complies with § 312k. The statutory transition rules matter: the March 2022 renewal-rule changes did not simply rewrite all pre-existing contracts.
- **Example:** A consumer signs up online for a subscription that automatically renews. If the contract is within § 312k, the consumer should be able to initiate cancellation through the required online process.
- **Official source:** [BMJ explanation of the phased changes and transitional treatment](https://hdr4.bmj.de/SharedDocs/Pressemitteilungen/DE/2022/0228_faire_Verbrauchervertraege.html); [BMJ legislative page](https://hdr4.bmj.de/SharedDocs/Gesetzgebungsverfahren/DE/2020_Faire_Verbrauchervertraege.html).

### 2.3 Mietrecht and housing

#### Reform: Mietspiegel

- **Relevant provisions:** §§ 558c and 558d BGB; transition provision in Art. 229 § 62 EGBGB.
- **Amending legislation:** *Gesetz zur Reform des Mietspiegelrechts (Mietspiegelreformgesetz – MsRG)*; enacted 10 August 2021, published **BGBl. 2021 I, p. 3515**; principal changes effective **1 July 2022**.
- **Previous law:** The statutory framework did not impose the same obligation on municipalities with at least 50,000 inhabitants to prepare a Mietspiegel and did not have the same detailed statutory and regulatory basis for preparation of a *qualifizierter Mietspiegel*.
- **Current law:** The reform strengthened the rules for preparing and updating Mietspiegel and introduced a duty for qualifying municipalities to prepare one. The EGBGB provided a transition period for initial preparation.
- **Purpose:** To improve the availability and consistency of rent-reference data used in rent-increase disputes.
- **Practical significance:** Mietspiegel remain an evidential and procedural aid in determining the *ortsübliche Vergleichsmiete*. The reform does not itself give each tenant an individual claim requiring the municipality to produce a particular Mietspiegel.
- **Example:** In a rent-increase dispute, a court may assess whether a cited Mietspiegel is qualified and whether its preparation complied with the applicable statutory and regulatory standards.
- **Official source:** [BMJ law text](https://www.bmjv.de/SharedDocs/Downloads/DE/Gesetzgebung/BGBl/Bgbl_Mietspiegelreformgesetz.pdf?__blob=publicationFile&v=3); [EGBGB transition provisions](https://www.gesetze-im-internet.de/bgbeg/).

#### Reform: Heating-system modernisation and rent increases

- **Relevant provisions:** §§ 555b no. 1a and 559e BGB.
- **Amending legislation:** *Gesetz zur Änderung des Gebäudeenergiegesetzes, zur Änderung des Bürgerlichen Gesetzbuches, zur Änderung der Verordnung über Heizkostenabrechnung, zur Änderung der Betriebskostenverordnung und zur Änderung der Kehr- und Überprüfungsordnung*; enacted 16 October 2023, published **BGBl. 2023 I No. 280**; relevant BGB provisions effective **1 January 2024**.
- **Previous law:** The BGB did not contain this specific category of modernisation measure and special rent-increase mechanism for installation of a heating system meeting the referenced GEG requirements.
- **Current law:** A qualifying heating-system installation is treated as a modernisation measure under § 555b no. 1a. Section 559e provides a special calculation route where qualifying public funding is used, including limits on the resulting rent increase.
- **Purpose:** To align tenancy law with the heating-system provisions of the GEG and specify how qualifying costs may be allocated.
- **Practical significance:** The landlord’s ability to pass costs through depends on statutory conditions, deductions and caps; the measure does not permit an unqualified transfer of the full installation cost to tenants.
- **Example:** A landlord replaces a heating system and receives qualifying public funding. The landlord must apply the special § 559e calculation and relevant limits, rather than simply treating all expenditure as an ordinary modernisation cost.
- **Official source:** [BGBl. 2023 I No. 280](https://www.recht.bund.de/bgbl/1/2023/280/VO.html).

### 2.4 Legal capacity, guardianship and representation

#### Reform: Vormundschafts- und Betreuungsrecht

- **Relevant provisions:** Among others §§ 1358, 1814, 1821, 1833 and 1862 BGB.
- **Amending legislation:** *Gesetz zur Reform des Vormundschafts- und Betreuungsrechts*; enacted 4 May 2021, published **BGBl. 2021 I, p. 882**; effective **1 January 2023**.
- **Previous law:** The prior statutory system was reorganised. The new framework makes the *Erforderlichkeitsgrundsatz* and the wishes of the person receiving Betreuung more explicit and central to the statutory duties of the Betreuer.
- **Current law:** A Betreuer is to support the person in conducting their affairs in accordance with their wishes; the Betreuer’s representative power is not a general substitute for the person’s own decision-making. The reform also added a limited emergency right of spousal representation in health matters under § 1358.
- **Purpose:** To strengthen self-determination, improve quality and supervision of Betreuung, and modernise Vormundschaft law.
- **Practical significance:** A court-appointed Betreuer must consider and, within statutory limits, follow the person’s wishes. The spouse’s emergency power is limited and does not amount to general authority over all personal or financial affairs.
- **Example:** A person with a Betreuer expresses a settled wish to remain in their home. The Betreuer’s decision-making and the court’s supervision must engage with that wish; the Betreuer cannot treat the appointment alone as authority to disregard it.
- **Transition:** The EGBGB contains transitional rules for existing appointments and related matters.
- **Official source:** [BMJ summary of the reform](https://www.bmj.de/SharedDocs/Pressemitteilungen/DE/2022/1229_Neues_Vormundschafts_und_Betreuungsrecht.html); [BMJ overview of Betreuung law](https://www.bmj.de/DE/themen/vorsorge_betreuungsrecht/rechtliche_betreuung/Rechtliche_Betreuung_Reform.html); [EGBGB transition rules](https://www.gesetze-im-internet.de/bgbeg/).

### 2.5 Property and business organisation: GbR reform

#### Reform: Modernisation of partnership law (MoPeG)

- **Relevant provisions:** Principally §§ 705 ff. BGB, including §§ 707 ff. and 713; further changes affect partnerships and related statutes.
- **Amending legislation:** *Gesetz zur Modernisierung des Personengesellschaftsrechts (Personengesellschaftsrechtsmodernisierungsgesetz – MoPeG)*; enacted 10 August 2021, published **BGBl. 2021 I, p. 3436**; most relevant changes effective **1 January 2024**.
- **Previous law:** The statutory BGB provisions did not fully reflect the case-law-recognised capacity of an *Außen-GbR* to participate in legal transactions and hold rights in its own name. The statutory framework retained a less suitable historical model for many operating partnerships.
- **Current law:** The BGB now expressly distinguishes the legally capable partnership from the non-legally-capable internal partnership and provides for a public *Gesellschaftsregister*. Registration is not universally compulsory, but it is practically required for an eligible GbR seeking to make certain entries in public registers, including the land register.
- **Purpose:** To bring partnership statutes into line with modern business practice and long-standing judicial recognition of the external GbR’s legal capacity.
- **Practical significance:** Existing and newly formed GbRs must assess whether they need registration, especially when acquiring or dealing with registered rights. The MoPeG also changed internal governance and dissolution rules.
- **Example:** A GbR wishing to be registered as owner of land must address the new registration framework; a property-owning GbR may need to register in the *Gesellschaftsregister* before the land-register transaction can proceed.
- **Official source:** [BGH library materials on the MoPeG](https://www.bundesgerichtshof.de/DE/Bibliothek/GesMat/WP19/P/PersonengesellschaftsRModG.html); [BGB current text](https://www.gesetze-im-internet.de/bgb/).

### 2.6 Commercial and other non-residential leases

#### Reform: Text form under § 578 BGB

- **Relevant provisions:** §§ 550 and 578 BGB; transition provision Art. 229 § 70 EGBGB.
- **Amending legislation:** *Viertes Gesetz zur Entlastung der Bürgerinnen und Bürger, der Wirtschaft sowie der Verwaltung von Bürokratie (Viertes Bürokratieentlastungsgesetz – BEG IV)*; enacted 23 October 2024, published **BGBl. 2024 I No. 323**; effective **1 January 2025**.
- **Previous law:** For leases of land or non-residential premises lasting more than one year, § 550 applied through § 578 with a written-form requirement; failure could mean the lease was treated as being for an indefinite period.
- **Current law:** For the leases covered by § 578, text form now suffices for the relevant § 550 rule. **Residential leases remain subject to the written-form rule.**
- **Purpose:** To reduce formal burdens in commercial and other non-residential leasing while preserving a documentary record of the agreement.
- **Practical significance:** Electronic documents and other forms meeting *Textform* may suffice for covered leases and amendments. The distinction between residential and non-residential leases remains essential.
- **Example:** For a qualifying commercial lease, a later agreement recorded in text form may avoid a written-form defect that could previously have affected the lease term.
- **Transition:** Under Art. 229 § 70 EGBGB, the old rule generally continued for pre-2025 leases through **1 January 2026**, subject to the provision’s rule for later agreed amendments.
- **Official source:** [BGBl. 2024 I No. 323](https://www.recht.bund.de/bgbl/1/2024/323/regelungstext.pdf); [EGBGB transition rule](https://www.gesetze-im-internet.de/bgbeg/).

### 2.7 2026 consumer and family-law changes

#### Reform: Online withdrawal function and consumer information

- **Relevant provisions:** §§ 312, 312j and 356a BGB; related EGBGB information provisions.
- **Amending legislation:** *Gesetz zur Änderung des Verbrauchervertrags- und des Versicherungsvertragsrechts sowie zur Änderung des Behandlungsvertragsrechts*; enacted 3 February 2026, published **BGBl. 2026 I No. 28**.
- **Entry into force:** The Act has staggered commencement dates. The principal consumer-contract changes took effect **19 June 2026**; the provisions specified for **27 September 2026** took effect on that date. The Act also contains provisions with other commencement dates.
- **Previous law:** The statutory online **cancellation** mechanism under § 312k was not an equivalent general online **withdrawal** mechanism for the consumer’s *Widerrufsrecht*.
- **Current law:** Section 356a establishes an electronic withdrawal function for covered distance contracts made through an online interface. The later commencement provisions also introduce or adjust consumer information duties, including information relevant to product guarantees and conformity rights.
- **Purpose:** To implement EU consumer-protection changes, make withdrawal rights easier to exercise online, and improve consumer information.
- **Practical significance:** Businesses enabling relevant online distance contracting need to provide the legally compliant withdrawal function and update customer-facing information. The separate regimes for cancellation and withdrawal should not be conflated.
- **Example:** A consumer concludes a distance contract through a retailer’s website and later exercises a statutory withdrawal right. The business must provide the prescribed online withdrawal route where § 356a applies.
- **Official source:** [BGBl. 2026 I No. 28](https://www.recht.bund.de/bgbl/1/2026/28/VO.html).

#### Reform: Acknowledgments of paternity and prevention of abusive recognition

- **Relevant provisions:** The reform changed BGB provisions concerning acknowledgment of paternity, including the former § 1597a BGB; associated amendments also concern other statutes.
- **Amending legislation:** *Gesetz zur besseren Verhinderung missbräuchlicher Anerkennungen der Vaterschaft und zur Änderung weiterer Vorschriften*; enacted 21 July 2026, published **BGBl. 2026 I No. 221**; effective **29 July 2026**.
- **Previous law:** The former § 1597a BGB provided a preventive procedure addressing suspected abusive acknowledgments, involving suspension and referral to the immigration authority in the circumstances covered by that provision.
- **Current law:** The former § 1597a BGB procedure was replaced. The new statutory scheme uses an administrative consent procedure in specified circumstances, principally under amended immigration-law provisions, with related changes to the BGB and other laws.
- **Purpose:** To revise the preventive framework used to address acknowledgments considered potentially abusive, while changing how the relevant screening and consent process operates.
- **Practical significance:** Notaries and other persons involved in recording acknowledgments must apply the new framework and its transitional provisions. The reform should not be mistaken for a general abolition of the rules on acknowledgment and contestation of paternity.
- **Transition:** The legislation includes transitional rules, including for cases already in progress under the former procedure.
- **Official source:** [BGBl. 2026 I No. 221](https://www.recht.bund.de/bgbl/1/2026/221/VO.html).

### BGB comparison table

| BGB provision(s) | Previous law | New law | Effective date |
|---|---|---|---|
| §§ 327–327u | No equivalent general consumer regime for digital products | Dedicated consumer rules for supply, conformity and remedies | 1 Jan 2022 |
| §§ 434, 475b–475e | No tailored consumer sales/update regime for goods with digital elements; shorter defect presumption | Tailored conformity and update rules; one-year presumption in consumer sales | 1 Jan 2022 |
| §§ 309 no. 9, 312k | Longer permitted AGB renewal/notice arrangements; no statutory online cancellation button | Shorter renewal and notice terms for covered new contracts; online cancellation function | 1 Mar / 1 Jul 2022 |
| §§ 558c–558d | Less developed statutory Mietspiegel framework | Expanded preparation rules and duty for qualifying municipalities | 1 Jul 2022 |
| §§ 1358, 1814, 1821, 1833 | Earlier Betreuung framework | Wishes-centred Betreuung; limited spousal emergency representation | 1 Jan 2023 |
| §§ 705 ff. | Statutory GbR framework did not fully reflect external GbR’s legal capacity | Modernised GbR framework and Gesellschaftsregister | 1 Jan 2024 |
| §§ 555b no. 1a, 559e | No specific heating-system modernisation and rent-allocation mechanism | Specific measure and conditional rent-increase route | 1 Jan 2024 |
| §§ 550, 578 | Written form applied to covered long non-residential leases | Text form generally suffices for covered leases; residential leases remain distinct | 1 Jan 2025, subject to transition |
| § 356a and related provisions | No general statutory online withdrawal function of this kind | Electronic withdrawal function and revised information rules | 19 Jun / 27 Sep 2026, by provision |
| Former § 1597a and related rules | Preventive procedure under former BGB provision | Replaced framework for specified cases; new statutory consent procedure | 29 Jul 2026 |

**Other BGB areas in the research scope.** No comparably broad enacted reform of BGB inheritance law or the general limitation-period regime was identified in this review. That does not mean no individual provision affecting those areas was amended. Changes to case law—such as judicial interpretations of limitation or damages rules—are not statutory amendments and are not counted here.

---

## 3. Changes to the German Criminal Code (StGB)

### 3.1 Offences relating to abortion information

#### Reform: Repeal of § 219a StGB

- **Relevant provisions:** Former § 219a StGB; consequential amendment to § 218b StGB.
- **Amending legislation:** *Gesetz zur Änderung des Strafgesetzbuches – Aufhebung des Verbots der Werbung für den Schwangerschaftsabbruch (§ 219a StGB)*, with related amendments; enacted 11 July 2022, published **BGBl. 2022 I No. 25**, including p. 1082; effective **19 July 2022**.
- **Previous law:** § 219a criminalised specified public advertising or informational conduct relating to abortion services.
- **Current law:** § 219a was repealed. The repeal did not abolish the separate offences and requirements relating to unlawful abortion or the statutory counselling framework.
- **Purpose:** To remove the specific criminal prohibition on public information about abortion services and the providers offering them.
- **Practical significance:** The offence no longer applies to post-commencement conduct. Related provisions were adjusted to reflect the repeal.
- **Example:** A doctor publicly provides factual information about the methods used in lawful abortion care. The former § 219a offence no longer applies merely because the information concerns abortion services.
- **Official source:** [Bundesrat legislative record, including BGBl publication details](https://www.bundesrat.de/SharedDocs/beratungsvorgaenge/2022/0201-0300/0290-22.html); [current StGB](https://www.gesetze-im-internet.de/stgb/).

### 3.2 Hate speech and statements about international crimes

#### Reform: New § 130(5) StGB

- **Relevant provisions:** § 130(5) StGB; renumbering of subsequent paragraphs.
- **Amending legislation:** *Gesetz zur Änderung des Bundeszentralregistergesetzes und des Strafgesetzbuches*; enacted 4 December 2022, **BGBl. 2022 I, p. 2146**; effective **9 December 2022**.
- **Previous law:** § 130 contained offences concerning incitement to hatred and certain forms of Holocaust denial or approval of Nazi rule. It did not contain this separate provision covering the specified public statements about offences under the Völkerstrafgesetzbuch.
- **Current law:** The new § 130(5) criminalises, subject to its statutory elements, publicly or at a meeting approving, denying or grossly trivialising an act of the kind described in §§ 6–12 VStGB, where the conduct is suitable to incite hatred or violence against the specified persons or groups **and** to disturb the public peace.
- **Purpose:** To extend the statutory treatment of certain public statements about international crimes beyond the provisions already in § 130.
- **Practical significance:** The provision is not a general ban on discussion, denial or criticism of historical or current events. The prosecution must establish the statutory conduct, manner, relevant target and required risk of incitement and disturbance of public peace.
- **Example:** A public speaker’s statement about an alleged international crime could only fall within § 130(5) if all statutory elements—including the specified form of approval, denial or gross trivialisation and the relevant suitability to incite and disturb public peace—are established.
- **Official source:** [Bundestag Scientific Services analysis identifying the new § 130(5)](https://www.bundestag.de/resource/blob/934836/74495c11da75f696859849cfa81118ad/WD-7-111-22-pdf-data.pdf); [current § 130 StGB](https://www.gesetze-im-internet.de/stgb/__130.html).

### 3.3 Sentencing and criminal sanctions

#### Reform: *Gesetz zur Überarbeitung des Sanktionenrechts*

- **Relevant provisions:** In particular §§ 43, 46(2), 56d and 67d StGB; related changes also affected procedural and enforcement law.
- **Amending legislation:** *Gesetz zur Überarbeitung des Sanktionenrechts – Ersatzfreiheitsstrafe, Strafzumessung, Auflagen und Weisungen sowie Unterbringung in einer Entziehungsanstalt*; enacted 26 July 2023, published **BGBl. 2023 I No. 203**. The principal StGB changes took effect **1 October 2023**.
- **Previous law:** In general, one day of *Ersatzfreiheitsstrafe* corresponded to one day-fine unit. The sentencing provision did not expressly list “gender-specific” motives alongside the specified discriminatory motives in the same way as the amended provision.
- **Current law:** One day of *Ersatzfreiheitsstrafe* generally corresponds to **two** day-fine units, reducing the custodial substitute for an unpaid fine. The sentencing provision expressly includes gender-specific and other inhuman motives among relevant sentencing considerations. The reform also adjusted requirements relating to *Unterbringung in einer Entziehungsanstalt* and supervision.
- **Purpose:** To reduce the custodial consequences of unpaid fines, make sentencing factors more explicit and revise the legal framework for treatment orders.
- **Practical significance:** A person unable to pay a fine faces a lower substitute-custody ratio than under the prior rule, although the fine remains payable and other enforcement measures may apply. Courts must also consider the amended sentencing language.
- **Example:** A person’s sentence is 60 day-fine units. Under the new conversion rule, the corresponding substitute imprisonment is generally 30 days, rather than the former 60 days.
- **Transition:** The Act contains transitional provisions, including provisions for specified pre-commencement treatment orders. A case involving an earlier order requires checking the relevant transition rule rather than applying the new provision automatically.
- **Official source:** [BGBl. 2023 I No. 203](https://www.recht.bund.de/bgbl/1/2023/203/VO.html); [2023 amending and transitional statute](https://www.recht.bund.de/bgbl/1/2023/218/VO.html).

### 3.4 Sexual offences involving child-pornographic content

#### Reform: Adjustment of minimum penalties under § 184b StGB

- **Relevant provisions:** § 184b(1) sentence 1 and § 184b(3) StGB.
- **Amending legislation:** *Gesetz zur Anpassung der Mindeststrafen des § 184b Absatz 1 Satz 1 und Absatz 3 des Strafgesetzbuches – Verbreitung, Erwerb und Besitz kinderpornographischer Inhalte*; enacted 24 June 2024, published **BGBl. 2024 I No. 213**; effective **28 June 2024**.
- **Previous law:** The 2021 tightening had set minimum penalties of one year’s imprisonment for the specified offences, making them *Verbrechen* under § 12(1) StGB.
- **Current law:** The minimum penalty was reduced to **six months** for the specified § 184b(1) conduct and to **three months** for § 184b(3). Those forms are no longer automatically *Verbrechen* by reason of a one-year minimum; their classification and available procedural consequences must be assessed under the revised ranges and the general rules.
- **Purpose:** To allow a more proportionate, individualised penalty response, including for less serious cases.
- **Practical significance:** Sentencing exposure and procedural options changed. The amendment also matters to older conduct through the *lex mitior* rule in § 2(3) StGB where the later law is more lenient.
- **Example:** A person charged with possession under § 184b(3) must be sentenced under the applicable law, including consideration of the reduced minimum where the amendment is the more lenient law.
- **Official source:** [BGBl. 2024 I No. 213](https://www.recht.bund.de/bgbl/1/2024/213/VO.html); [current § 184b StGB](https://www.gesetze-im-internet.de/stgb/__184b.html).

### 3.5 Economic and political crime

#### Reform: New offence of impermissible interest representation

- **Relevant provisions:** § 108f StGB; related amendment to § 5 StGB and to jurisdictional provisions in the GVG.
- **Amending legislation:** *Gesetz zur Änderung des Strafgesetzbuches – Strafbarkeit der unzulässigen Interessenwahrnehmung*; enacted 12 June 2024, published **BGBl. 2024 I No. 190**; effective **18 June 2024**.
- **Previous law:** The StGB had no § 108f offence covering the specified paid interest-representation conduct. § 108e addressed a different offence concerning bribery and corruption of members of representative bodies.
- **Current law:** Section 108f criminalises, subject to its precise statutory elements, demanding, accepting or having promised an unjustified pecuniary benefit as consideration for specified conduct in the interests of the benefit provider or another person during the mandate, where the arrangement violates rules governing the office-holder’s legal position. The offence applies only to the categories of mandate-holder specified in the statute.
- **Purpose:** To address specified commercialisation of political influence outside the existing § 108e offence.
- **Practical significance:** The offence is not a general prohibition on lobbying or political advocacy. A prosecution must establish the unjustified benefit, the consideration relationship, covered office-holder status and violation of the relevant rules.
- **Example:** A covered mandate-holder accepts an unjustified payment in exchange for carrying out specified external interest representation, in circumstances that violate the rules governing that mandate. Section 108f may be relevant if its elements are met.
- **Official source:** [BGBl. 2024 I No. 190, including § 108f text](https://www.bundestag.de/resource/blob/1011214/BGBl-2024-I-Nr-190.pdf); [current StGB](https://www.gesetze-im-internet.de/stgb/).

### 3.6 Terrorism and foreign intelligence activity

#### Reform: Implementation of Directive (EU) 2017/541

- **Relevant provisions:** In particular §§ 6, 89a and 99 StGB; related provisions include §§ 89c, 126 and 129a.
- **Amending legislation:** *Gesetz zur Umsetzung der Richtlinie (EU) 2017/541 zur Terrorismusbekämpfung und zur Anpassung des Strafrahmens bei geheimdienstlicher Agententätigkeit*; enacted 20 March 2026, published **BGBl. 2026 I No. 95**; effective **2 April 2026**.
- **Previous law:** Section 89a used the framework of a *schwere staatsgefährdende Gewalttat* and a narrower catalogue of predicate offences and preparation conduct.
- **Current law:** The statute revised the terrorism framework and preparation offence to implement the EU directive, including revised definitions and offences within the statutory scheme. It also changed the penalty framework for secret-service agent activity under § 99. Consequential provisions were adjusted to refer to the revised offences.
- **Purpose:** To align German law with Directive (EU) 2017/541 and amend the treatment of secret-service agent activity.
- **Practical significance:** The scope of potential liability for preparation and related conduct requires close attention to the amended statutory elements. The amendments do not eliminate the need to prove the specific conduct and intent required by the relevant provision.
- **Example:** A case concerning travel or preparation allegedly connected to a terrorist offence must be analysed under the post-2 April 2026 wording of § 89a, not by simply carrying forward the former statutory terminology.
- **Official source:** [BGBl. 2026 I No. 95](https://www.recht.bund.de/bgbl/1/2026/95/VO.html); [current StGB](https://www.gesetze-im-internet.de/stgb/).

### StGB comparison table

| StGB provision(s) | Previous law | New law | Effective date |
|---|---|---|---|
| § 219a (repealed) | Criminalised specified public information/advertising about abortion services | Offence repealed; separate abortion rules remain | 19 Jul 2022 |
| § 130(5) | No separate offence in this form for the specified statements about international crimes | New offence subject to statutory incitement and public-peace elements | 9 Dec 2022 |
| §§ 43, 46(2), 56d, 67d | One day-fine unit generally equalled one day of substitute imprisonment; prior sentencing and treatment wording | Two day-fine units generally equal one day; sentencing and treatment rules revised | 1 Oct 2023 |
| § 184b(1), (3) | One-year minimum made specified offences *Verbrechen* | Minimums reduced to six months and three months respectively | 28 Jun 2024 |
| § 108f (new) | No offence in this form | Criminalises specified impermissible paid interest representation by covered mandate-holders | 18 Jun 2024 |
| §§ 6, 89a, 89c, 99 and related references | Earlier terrorism-preparation framework and penalty provisions | Revised terrorism framework and related rules; § 99 penalty changes | 2 Apr 2026 |

**Other StGB areas in the research scope.** No substantial StGB amendment to general criminal-liability principles, cybercrime offences, or the core sexual-offence provisions is identified above as a principal reform in this review. Drug-law changes should not be attributed to the StGB merely because criminal penalties are involved: the principal cannabis reform was enacted in the **Konsumcannabisgesetz (KCanG)**, not as a general amendment to the StGB. Similarly, changes in the *Völkerstrafgesetzbuch* or the StPO are not StGB amendments unless the amending statute actually changes StGB provisions.

---

## 4. Timeline of Legislative Changes

### 2022
- **1 January:** Digital-products consumer rules and revised consumer sales law take effect.
- **1 March:** Fair-consumer-contract rules on AGB renewals and notice periods take effect for covered new contracts.
- **1 July:** Online cancellation function under § 312k BGB takes effect for covered contracts.
- **1 July:** Principal Mietspiegel reform provisions take effect.
- **19 July:** § 219a StGB is repealed.
- **9 December:** New § 130(5) StGB takes effect.

### 2023
- **1 January:** Reform of Vormundschaft and Betreuung takes effect, including § 1358 BGB.
- **1 October:** Principal sanctions-law reforms take effect, including the changed *Ersatzfreiheitsstrafe* conversion rule.

### 2024
- **1 January:** MoPeG and the specific heating-system tenancy-law provisions take effect.
- **28 June:** Reduced § 184b StGB minimum penalties take effect.
- **18 June:** New § 108f StGB takes effect.  
  *The order here reflects effective dates, not the sequence of publication.*
- **2 August:** The *Gesetz zur Fortentwicklung des Völkerstrafrechts* is published. Its principal changes concern the VStGB and related law; it should not be misreported as a general StGB reform.
- **23 October:** BEG IV is enacted; its relevant BGB text-form change takes effect in 2025.

### 2025
- **1 January:** § 578 BGB text-form reform takes effect for covered non-residential leases, subject to transition rules.
- **12 December:** The *Gesetz zur Einführung einer elektronischen Präsenzbeurkundung* is published. It amended, among other laws, §§ 126 and 129 BGB; its commencement was tied to the statute’s entry-into-force provision. This is principally a form and notarial-process reform, rather than a substantive expansion of ordinary contract remedies.

### 2026 (up to 2 November)
- **2 April:** Revised terrorism and secret-service-agent provisions take effect.
- **19 June and 27 September:** Staggered consumer-contract changes under BGBl. 2026 I No. 28 take effect, including the online withdrawal function and later consumer-information provisions.
- **29 July:** New parentage-recognition framework takes effect, replacing the former § 1597a BGB approach.

---

## 5. Practical Implications

### Private individuals
- Consumers should distinguish **digital-product** claims from claims about physical goods and goods with digital elements.
- Subscribers should check whether renewal clauses and online cancellation options comply with the applicable version of the BGB and its transition rules.
- People who may need legal support should understand that Betreuung is not a blanket transfer of decision-making: the person’s wishes are central, subject to statutory limits.
- Parties to non-residential leases should record agreements in text form and check whether a legacy lease remains within the EGBGB transition period.
- People involved in criminal proceedings must consider when conduct occurred and whether a later, more lenient law applies under § 2(3) StGB.

### Businesses
- Online sellers and service providers should review digital-product terms, update obligations, defect-handling procedures and consumer-facing information.
- Subscription businesses should check both AGB renewal terms and the practical operation of cancellation functions.
- Landlords and tenants should carefully distinguish residential from non-residential leases and apply the appropriate form and transition rules.
- Businesses using a GbR to hold property or participate in registers should assess the MoPeG registration framework.
- Online businesses should assess whether § 356a BGB applies to their distance-contract process from the relevant 2026 commencement date.

### Lawyers and legal practitioners
- Identify the date of the relevant contract, act, decision or proceeding before selecting the statutory version.
- Check *Übergangsvorschriften* rather than assuming a new rule applies to every pre-existing relationship.
- For criminal cases, separately consider the offence’s classification as *Verbrechen* or *Vergehen*, available procedural routes and the *lex mitior* principle.
- Avoid treating amendments to the KCanG, VStGB, StPO or other statutes as amendments to the StGB or BGB.

### Courts and criminal justice authorities
- Courts must apply the correct temporal version of the statute and any transitional rules, particularly in sales, tenancy and criminal sentencing cases.
- Betreuung courts must apply the reformed wishes-centred supervisory framework.
- Criminal justice authorities should account for the revised § 184b penalty ranges and the changed substitute-imprisonment conversion.
- The new §§ 108f and 89a frameworks require element-by-element application rather than reliance on broad policy labels such as “lobbying” or “terrorism.”

---

## 6. Most Relevant Changes for German Legal Examinations

> **Confirmed requirements versus assessment:** There is no single nationwide statutory list declaring these reforms compulsory examination subjects for every Land or every examination sitting. The following “likely relevance” is an assessment based on the reforms’ doctrinal importance and practical case potential—not a claim about a confirmed examination syllabus. Candidates should check the applicable Land’s current examination rules and official syllabus.

### Civil law

- **Digital-products and sales law — likely high relevance.** §§ 327 ff. and §§ 434, 475b ff. invite classification problems: digital service versus goods with digital elements; applicable conformity standard; update duties; and remedies.
- **Betreuung and § 1358 — likely high relevance.** The reform engages autonomy, capacity, representation, the relationship between assistance and substitution, and statutory limits on spousal emergency representation.
- **MoPeG — likely high relevance, especially commercial/private law.** The GbR’s legal capacity, property ownership, registration and internal governance are natural exam issues. Candidates must use the post-1 January 2024 statutory structure where applicable.
- **Commercial lease form — likely useful.** The distinction between *Schriftform* and *Textform*, residential and non-residential leases, and the transition rule provide a practical statutory-version issue.
- **Consumer renewal, cancellation and withdrawal — likely practical relevance.** A problem may ask candidates to distinguish § 312k cancellation from statutory withdrawal under § 356a and to identify the correct commencement date.
- **Limitation periods, inheritance and general damages — no comparable broad reform identified in this review.** Their continuing exam importance comes from the general law and case law, not a major statutory amendment established here.

### Criminal law

- **§ 184b adjustment — likely high relevance.** It tests offence classification, sentencing ranges, transitional law and § 2(3) StGB’s *lex mitior* principle.
- **Sanctions reform — likely high relevance.** The amended conversion between *Geldstrafe* and *Ersatzfreiheitsstrafe* and the treatment of statutory sentencing factors are directly practical.
- **§ 130(5) — likely relevant to advanced cases.** It requires careful parsing of the offence’s conduct, target, suitability to incite hatred or violence, and public-peace element. It should not be reduced to a generic “denial” offence.
- **§ 108f — likely useful for special-part and statutory-interpretation questions.** The limits on covered persons, unjustified advantage, consideration and breach of office-related rules matter.
- **2026 terrorism amendments — potentially important but specialised.** Candidates should use the exact post-amendment text where a case concerns § 89a or related offences, and avoid importing prior terminology without checking it.

### Procedural and practical questions

The most useful examination skill across these reforms is **temporal statutory analysis**: identify the date of the conduct or contract, check commencement and transitional clauses, and distinguish substantive amendments from changes to procedure or another statute. In criminal cases, this includes analysing whether a later law is more lenient; in civil cases, it often means deciding which statutory version governs an existing contract.

---

## 7. Conclusion

Since 2022, the most substantial BGB changes have concerned digital consumer transactions, the modernisation of the GbR, Betreuung and representation, and selected tenancy and online-consumer rules. The most important StGB changes have been targeted rather than a general rewrite of criminal liability: abortion-information law, public statements about international crimes, sentencing and substitute imprisonment, § 184b penalties, impermissible paid interest representation, and the 2026 terrorism framework.

For legal practice and examinations, the recurring lesson is to check the **specific amending statute, commencement date and transitional provision**. Not every related development is an amendment to either code: several prominent changes arose instead under the KCanG, VStGB, StPO or other legislation.

## 8. Official Sources and References

### Official codes and transition law
- [Bürgerliches Gesetzbuch (BGB), Gesetze im Internet](https://www.gesetze-im-internet.de/bgb/)
- [Einführungsgesetz zum Bürgerlichen Gesetzbuche (EGBGB), including transition rules](https://www.gesetze-im-internet.de/bgbeg/)
- [Strafgesetzbuch (StGB), Gesetze im Internet](https://www.gesetze-im-internet.de/stgb/)
- [Bundesgesetzblatt, official publication platform](https://www.recht.bund.de/)
- [Gesetze im Internet note on authoritative BGBl publication](https://www.gesetze-im-internet.de/aktuDienst.html)

### Principal legislation and official materials
- [Digital-products law, BGBl. 2021 I, p. 2123 — BMJ-hosted law text](https://www.bmjv.de/SharedDocs/Downloads/DE/Gesetzgebung/BGBl/Bgbl_Digitale_Inhalte.pdf?__blob=publicationFile&v=3)
- [Sales of goods with digital elements — BMJ-hosted law text](https://www.bmjv.de/SharedDocs/Downloads/DE/Gesetzgebung/BGBl/Bgbl_Regelung_des_Verkaufs_von_Sachen_mit_digitalen_Elementen.html)
- [Fair consumer contracts — BMJ explanation](https://hdr4.bmj.de/SharedDocs/Pressemitteilungen/DE/2022/0228_faire_Verbrauchervertraege.html)
- [Mietspiegelreformgesetz — BMJ-hosted law text](https://www.bmjv.de/SharedDocs/Downloads/DE/Gesetzgebung/BGBl/Bgbl_Mietspiegelreformgesetz.pdf?__blob=publicationFile&v=3)
- [Betreuung reform — BMJ explanation](https://www.bmj.de/SharedDocs/Pressemitteilungen/DE/2022/1229_Neues_Vormundschafts_und_Betreuungsrecht.html)
- [MoPeG — BGH library materials](https://www.bundesgerichtshof.de/DE/Bibliothek/GesMat/WP19/P/PersonengesellschaftsRModG.html)
- [Heating-system tenancy amendments — BGBl. 2023 I No. 280](https://www.recht.bund.de/bgbl/1/2023/280/VO.html)
- [BEG IV — BGBl. 2024 I No. 323](https://www.recht.bund.de/bgbl/1/2024/323/regelungstext.pdf)
- [Abortion-advertising repeal — Bundesrat legislative record and BGBl details](https://www.bundesrat.de/SharedDocs/beratungsvorgaenge/2022/0201-0300/0290-22.html)
- [§ 130(5) analysis — Bundestag Scientific Services](https://www.bundestag.de/resource/blob/934836/74495c11da75f696859849cfa81118ad/WD-7-111-22-pdf-data.pdf)
- [Sanktionenrechtsgesetz — BGBl. 2023 I No. 203](https://www.recht.bund.de/bgbl/1/2023/203/VO.html)
- [§ 184b penalties — BGBl. 2024 I No. 213](https://www.recht.bund.de/bgbl/1/2024/213/VO.html)
- [§ 108f StGB — law text hosted by the Bundestag](https://www.bundestag.de/resource/blob/1011214/BGBl-2024-I-Nr-190.pdf)
- [Terrorism and secret-service-agent reform — BGBl. 2026 I No. 95](https://www.recht.bund.de/bgbl/1/2026/95/VO.html)
- [2026 consumer-contract reform — BGBl. 2026 I No. 28](https://www.recht.bund.de/bgbl/1/2026/28/VO.html)
- [2026 paternity-recognition reform — BGBl. 2026 I No. 221](https://www.recht.bund.de/bgbl/1/2026/221/VO.html)

In [38]:
with open("german_law_reforms_2022_2026.md", "w", encoding="utf-8") as f:
    f.write(final_answer_4)

print("Report saved successfully!")

Report saved successfully!
